# Segmentation **cascade** vs depth-only vs control — 5-fold CV on NYU Depth V2 — **DA V2 Relative Small**

Two stages in one notebook:

| stage | what it does | why |
|---|---|---|
| **1. Fine-tune** | head-only SILog fine-tune of `depth-anything/Depth-Anything-V2-Small-hf` on `train[0:1000]` | convert it from relative to **metric** |
| **2. Cascade evaluation** | three-arm ablation on `train[1000:3000]`, a **disjoint** slice | answers the thesis question |

**Leakage is the reason for the split.** Fine-tuning on NYU train would normally force evaluation onto the 654-image validation split. Instead the train split is carved in two — the model trains on the first `1000` images and is scored on the next `2000`, which it never saw. Disjoint by construction, and large enough to fold.

**Why two stages instead of a zero-shot comparison.** `depth-anything/Depth-Anything-V2-Small-hf` is a **relative** checkpoint: it emits unitless disparity, which the depth metrics cannot score at all — there is no metres anywhere in its output. Fine-tuning with SILog (a scale-invariant log loss) against metric ground truth is what turns it into a metric predictor. The zero-shot alternative is a per-image scale+shift fit against ground truth, which leaks GT information into the prediction; fine-tuning does not, so this stage exists to make the comparison honest as well as possible.

## The three arms

| arm | what the depth model sees |
|---|---|
| **base** | the original image (depth-only reference) |
| **casc** | the image with everything outside the SAM2 union filled gray |
| **ctrl** | the image with a **rolled copy** of the union filled gray |

**Why the mask is baked into the pixels.** Depth Anything V2 and MiDaS are monocular **RGB** estimators: they accept an image and nothing else — no mask input, no prompt. So the segmentation output cannot literally be handed to the depth model. The only training-free way to make segmentation "feed" it is to change the pixels: everything outside the SAM2 union becomes neutral gray. The depth model still only ever sees RGB.

## Why the control arm decides whether this experiment means anything

Masking changes **two** things at once: *which* region is hidden, and *how many* pixels are hidden. A cascade that beats the depth-only baseline may be exploiting SAM2's **semantics** — or merely reacting to a large flat gray patch, because these models lean on global context and any big occlusion perturbs them.

The control separates them. `roll(union)` has **exactly the same area and shape** as the real union and differs only in *where* it sits. So:

- `casc vs base` → the total effect of the cascade
- `ctrl vs base` → the effect of masking that much area, with **no semantics at all**
- **`casc vs ctrl` → the part attributable to segmentation** ← the actual thesis question

**If `casc ≈ ctrl`, the "cascade" is a masking artefact, not segmentation.** The per-image control IoU is recorded and plotted so you can see when the control was too weak to be informative.

## Read this before quoting any number: coverage is the main risk

SAM2's automatic generator proposes a mask for essentially **every surface**, so the union can cover **85–95% of the frame**. At that point almost nothing is masked out, `casc ≈ base`, and the experiment answers nothing. The run reports **mean union coverage first**; `seg_cascade_kfold.py` exposes `--min-mask-area` / `--max-coverage` to drop the largest masks (walls, floor, ceiling) and re-run.

> **Comparability.** These numbers are for a **fine-tuned** checkpoint, so they are **not comparable** to the zero-shot `com1` notebooks — different weights *and* a different image set. The three cascade arms remain internally comparable, since all three share one checkpoint and one image set.

**Before running:** enable **GPU** (Settings → Accelerator → GPU T4/P100) and **Internet** (phone-verified Kaggle account).

**Runtime, and the real constraint.** Fine-tuning is ~1500 steps and the evaluation runs the depth model **three times per image** (plus SAM2 once). Together these can exceed a Kaggle session. **Both stages are resumable**: the fine-tune writes a checkpoint every `SAVE_EVERY` steps, and the evaluation appends each image to `cascade_per_sample.jsonl`. Download outputs before the session ends — Kaggle wipes `/kaggle/working`.

## 0. Install dependencies (only what's missing)

In [ ]:
import importlib.util, subprocess, sys
from importlib.metadata import version, PackageNotFoundError


def pkg_version(pkg):
    try:
        return tuple(int(p) for p in version(pkg).split(".")[:2])
    except PackageNotFoundError:
        return (0, 0)


# transformers>=4.48 is a safe floor for the depth-estimation pipeline. Do NOT
# upgrade numpy/opencv/scipy on Kaggle -- the prebuilt stack is a matched set
# and breaks if those change.
if pkg_version("transformers") < (4, 48):
    print("upgrading transformers...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.48"], check=True)
    print("transformers upgraded -- RESTART the session (Run > Restart & Clear), then re-run all cells.")

if pkg_version("datasets") == (0, 0):
    print("installing datasets...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)

# SAM2 (installing a NEW package needs no restart, unlike upgrading one).
if importlib.util.find_spec("sam2") is None:
    print("installing SAM2 (a few minutes)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/facebookresearch/sam2.git"], check=True)

# datasets>=4 removed loading-script support, so sayakpaul/nyu_depth_v2 is read
# through its auto-converted parquet branch. That branch is PARTIAL and holds
# only 3,100 train images -- stage 1 measures it and refuses to overrun it.
import transformers, datasets
print("transformers:", transformers.__version__, "| datasets:", datasets.__version__,
      "| sam2 available")

## 1. Write the scripts (all three)

Two files, because the cascade module **imports** the existing one rather than copying it:

- `seg_depth_kfold.py` — the com1 late-fusion module. Reused here for its metrics, NYU loader, SAM2 mask generator and fold assignment.
- `seg_cascade_kfold.py` — the three-arm cascade evaluation, plus the fine-tune leakage guard (it reads `ft_meta.json` from the checkpoint directory and **refuses to run** if the offset would score images the model trained on).

Writing both keeps the notebook self-contained.

In [ ]:
%%writefile seg_depth_kfold.py
#!/usr/bin/env python3
"""
seg_depth_kfold.py

5-FOLD CROSS-VALIDATION evaluation of the SEGMENTATION+DEPTH fusion model
(SAM2 masks + metric Depth Anything V2, object-median fusion) against the
SAME depth model WITHOUT segmentation, on NYU Depth V2.

This is the k-fold version of section 6 of compare_depth_models_kaggle.ipynb.
Instead of a single 100-image run it evaluates on --num-samples (default 3000)
images partitioned into 5 disjoint folds, and reports every metric as a
mean +/- std ACROSS the folds -- so the seg-vs-depth difference comes with a
measure of how stable it is, not a single point estimate.

IMPORTANT -- this is cross-validation for EVALUATION, not weight training.
Neither the depth model nor the object-median fusion rule has trainable
parameters here. Each image is scored exactly once (it belongs to one fold);
folds are then aggregated. "Fine-tuning with fivefold" here means the k-fold
evaluation protocol, giving a cross-validated mean+/-std comparison.

WHY THE TRAIN SPLIT
-------------------
The NYU-V2 validation split only has ~654 images -- not enough for 3000
samples, so we draw from the TRAIN split. This is still a fair test for the
depth backbone used here (Depth-Anything-V2-Metric-Indoor-*, trained on
Hypersim, NOT on NYU), so NYU train images are unseen data for it. Do NOT
reuse this protocol with an NYU-trained model (e.g. ZoeDepth-nyu) -- that
would be train/test leakage.

USAGE
-----
python seg_depth_kfold.py --num-samples 3000 --n-folds 5 --baseline da2-metric-small
python seg_depth_kfold.py --num-samples 200            # quick smoke test
python seg_depth_kfold.py --sam2-quick-test            # sparser/faster SAM2 grid

Resumable: per-sample scores are appended to <output-dir>/kfold_persample.jsonl
as each image finishes; re-running the same command skips images already
scored (essential -- 3000 images with SAM2 is long and Kaggle sessions die).
"""

import argparse
import json
import os
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import cv2

METRIC_FIELDS = ["abs_rel", "sq_rel", "rmse", "rmse_log", "delta1", "delta2", "delta3"]
LOWER_BETTER = {"abs_rel", "sq_rel", "rmse", "rmse_log"}

# Depth-only baselines. Slugs match compare_depth_models.py so results stay
# comparable to the 7-model notebook.
#   metric   checkpoints -> meters directly, scored as-is.
#   relative checkpoints -> unitless disparity (higher = closer), NOT meters;
#                           use relative=True so each prediction gets a per-image
#                           least-squares scale+shift alignment to GT inverse
#                           depth (MiDaS / DA-V2 protocol) before scoring. See
#                           align_relative_depth().
BASELINE_IDS = {
    "da2-metric-small": "depth-anything/Depth-Anything-V2-Metric-Indoor-Small-hf",
    "da2-metric-base":  "depth-anything/Depth-Anything-V2-Metric-Indoor-Base-hf",
    "da2-metric-large": "depth-anything/Depth-Anything-V2-Metric-Indoor-Large-hf",
    "da2-relative-small": "depth-anything/Depth-Anything-V2-Small-hf",
    "da2-relative-base":  "depth-anything/Depth-Anything-V2-Base-hf",
    "da2-relative-large": "depth-anything/Depth-Anything-V2-Large-hf",
}

# Slugs whose checkpoints emit relative disparity -> need scale+shift alignment.
RELATIVE_SLUGS = {"da2-relative-small", "da2-relative-base", "da2-relative-large"}

# SAM2.1 model-size presets: checkpoint URL, config path, filename.
SAM2_PRESETS = {
    "tiny": dict(
        url="https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt",
        config="configs/sam2.1/sam2.1_hiera_t.yaml", filename="sam2.1_hiera_tiny.pt"),
    "small": dict(
        url="https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt",
        config="configs/sam2.1/sam2.1_hiera_s.yaml", filename="sam2.1_hiera_small.pt"),
    "base_plus": dict(
        url="https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_base_plus.pt",
        config="configs/sam2.1/sam2.1_hiera_b+.yaml", filename="sam2.1_hiera_base_plus.pt"),
    "large": dict(
        url="https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt",
        config="configs/sam2.1/sam2.1_hiera_l.yaml", filename="sam2.1_hiera_large.pt"),
}


# =========================================================================
# Standard monocular depth metrics (Eigen et al.; same as compare_depth_models)
# =========================================================================
def compute_depth_metrics(pred: np.ndarray, gt: np.ndarray,
                          min_depth: float = 1e-3, max_depth: float = 10.0) -> Optional[Dict[str, float]]:
    """AbsRel/SqRel/RMSE/RMSE-log (lower better) + delta1/2/3 (higher better).
    Returns None if the valid-pixel mask is empty."""
    valid = ((gt > min_depth) & (gt < max_depth) & np.isfinite(gt)
             & np.isfinite(pred) & (pred > min_depth))
    pv, gv = pred[valid], gt[valid]
    if pv.size == 0:
        return None
    ratio = np.maximum(pv / gv, gv / pv)
    return {
        "abs_rel": float(np.mean(np.abs(pv - gv) / gv)),
        "sq_rel": float(np.mean(((pv - gv) ** 2) / gv)),
        "rmse": float(np.sqrt(np.mean((pv - gv) ** 2))),
        "rmse_log": float(np.sqrt(np.mean((np.log(pv) - np.log(gv)) ** 2))),
        "delta1": float(np.mean(ratio < 1.25)),
        "delta2": float(np.mean(ratio < 1.25 ** 2)),
        "delta3": float(np.mean(ratio < 1.25 ** 3)),
    }


def clip_prediction(pred: np.ndarray, min_depth: float = 1e-3, max_depth: float = 10.0) -> np.ndarray:
    return np.clip(pred, min_depth, max_depth)


def align_relative_depth(pred_disp: np.ndarray, gt_depth: np.ndarray,
                         min_depth: float = 1e-3, max_depth: float = 10.0):
    """Scale+shift alignment for relative-depth (disparity) models -- the
    standard MiDaS / Depth-Anything eval protocol, copied from
    compare_depth_models.align_relative_depth so behaviour matches the 7-model
    comparison exactly.

    Fits s, t so that s*pred_disp + t ~= 1/gt_depth over valid pixels (least
    squares), then returns depth = 1/(s*pred_disp + t) in meters. The fit uses
    ground-truth information, which is why aligned baselines are marked
    "(aligned)" in the notebook and are not directly comparable to metric models.

    Returns (aligned_depth_meters, s, t).
    """
    valid = ((gt_depth > min_depth) & (gt_depth < max_depth)
             & np.isfinite(gt_depth) & np.isfinite(pred_disp))
    if valid.sum() < 2:
        raise ValueError("Not enough valid pixels to fit scale+shift alignment.")

    d_gt = 1.0 / gt_depth[valid].astype(np.float64)
    p = pred_disp[valid].astype(np.float64)
    A = np.stack([p, np.ones_like(p)], axis=1)
    (s, t), *_ = np.linalg.lstsq(A, d_gt, rcond=None)

    aligned_disp = np.clip(s * pred_disp.astype(np.float64) + t, 1e-6, None)
    return (1.0 / aligned_disp).astype(np.float32), float(s), float(t)


# =========================================================================
# NYU Depth V2 loading (streaming -- never holds all images in memory) and
# the depth model. Mirrors eval_nyu_depth.py (incl. the datasets>=4 parquet
# fallback) so behaviour matches the depth-only baseline exactly.
# =========================================================================
def iter_nyu(split: str, num_samples: int):
    """Yields (idx, hf_item) lazily. Conversion to numpy happens in the caller
    so already-scored indices can be skipped cheaply on a resumed run."""
    from datasets import load_dataset
    try:
        ds = load_dataset("sayakpaul/nyu_depth_v2", split=split, streaming=True)
    except RuntimeError:
        data_files = {split: f"hf://datasets/sayakpaul/nyu_depth_v2@refs/convert/parquet/default/partial-{split}/*.parquet"}
        ds = load_dataset("parquet", data_files=data_files, split=split, streaming=True)
    for i, item in enumerate(ds):
        if i >= num_samples:
            break
        yield i, item


def load_metric_depth_model(model_id: str, device: str):
    from transformers import pipeline
    return pipeline(task="depth-estimation", model=model_id, device=0 if device == "cuda" else -1)


def predict_depth_meters(depth_pipe, image_rgb: np.ndarray) -> np.ndarray:
    """Predicted depth in meters (metric checkpoint), resized to the image."""
    from PIL import Image
    result = depth_pipe(Image.fromarray(image_rgb))
    pred = result["predicted_depth"].squeeze().detach().cpu().numpy().astype(np.float32)
    if pred.shape != image_rgb.shape[:2]:
        pred = cv2.resize(pred, (image_rgb.shape[1], image_rgb.shape[0]), interpolation=cv2.INTER_CUBIC)
    return pred


# =========================================================================
# SAM2 automatic mask generator (mirrors sam2_depth_pipeline.py)
# =========================================================================
def _download_with_progress(url: str, dest_path: str):
    import urllib.request
    def _hook(block_num, block_size, total_size):
        if total_size <= 0:
            return
        downloaded = block_num * block_size
        pct = min(100, downloaded * 100 // total_size)
        print(f"\r  downloading {os.path.basename(dest_path)}: {pct}% "
              f"({downloaded/1e6:.0f}/{total_size/1e6:.0f} MB)", end="", flush=True)
    urllib.request.urlretrieve(url, dest_path, reporthook=_hook)
    print()


def ensure_sam2_checkpoint(model_size: str, checkpoints_dir: str) -> str:
    preset = SAM2_PRESETS[model_size]
    os.makedirs(checkpoints_dir, exist_ok=True)
    dest = os.path.join(checkpoints_dir, preset["filename"])
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        print(f"Using cached SAM2 checkpoint: {dest}")
        return dest
    print(f"Downloading SAM2 checkpoint ({model_size})...")
    _download_with_progress(preset["url"], dest)
    return dest


def load_sam2_mask_generator(sam2_checkpoint: str, sam2_config: str, device: str, quick_test: bool = False):
    from sam2.build_sam import build_sam2
    from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator
    sam2_model = build_sam2(sam2_config, sam2_checkpoint, device=device, apply_postprocessing=True)
    return SAM2AutomaticMaskGenerator(
        model=sam2_model,
        points_per_side=16 if quick_test else 32,
        pred_iou_thresh=0.86,
        stability_score_thresh=0.92,
        crop_n_layers=0 if quick_test else 1,
        crop_n_points_downscale_factor=2,
        min_mask_region_area=100,
    )


def generate_masks(mask_generator, image_rgb: np.ndarray) -> List[Dict[str, Any]]:
    return sorted(mask_generator.generate(image_rgb), key=lambda m: m["area"], reverse=True)

# =========================================================================
# Object-median fusion (mirrors eval_segmentation_depth.py)
# =========================================================================
def _fit_mask(seg: np.ndarray, shape: Tuple[int, int]) -> np.ndarray:
    if seg.shape[:2] != shape:
        seg = cv2.resize(seg.astype(np.uint8), (shape[1], shape[0]),
                         interpolation=cv2.INTER_NEAREST).astype(bool)
    return seg


def fuse_object_median(depth: np.ndarray, mask_dicts: List[Dict[str, Any]]) -> np.ndarray:
    """Replace every pixel inside a mask by the median of the RAW depth inside
    that mask. Largest masks first, so smaller (more specific) segments win on
    overlap; the median always comes from the original depth (no contamination)."""
    fused = depth.copy()
    for m in sorted(mask_dicts, key=lambda m: m["area"], reverse=True):
        seg = _fit_mask(m["segmentation"].astype(bool), depth.shape[:2])
        vals = depth[seg]
        if vals.size == 0:
            continue
        fused[seg] = np.median(vals)
    return fused


def union_masks(mask_dicts: List[Dict[str, Any]], shape: Tuple[int, int]) -> np.ndarray:
    union = np.zeros(shape, dtype=bool)
    for m in mask_dicts:
        union |= _fit_mask(m["segmentation"].astype(bool), shape)
    return union


def metrics_in_region(pred: np.ndarray, gt: np.ndarray, region: np.ndarray,
                      max_depth: float = 10.0) -> Optional[Dict[str, float]]:
    if not region.any():
        return None
    pred_r = np.where(region, pred, 0.0).astype(np.float32)
    gt_r = np.where(region, gt, 0.0).astype(np.float32)
    return compute_depth_metrics(pred_r, gt_r, max_depth=max_depth)


# =========================================================================
# Score one image: baseline (depth-only) vs fused (seg+depth), whole-image
# and inside/outside-mask, plus mask coverage.
# =========================================================================
def score_sample(base_pred: np.ndarray, gt: np.ndarray,
                 masks: List[Dict[str, Any]], max_depth: float) -> Optional[Dict[str, Any]]:
    fused = fuse_object_median(base_pred, masks)
    base_c = clip_prediction(base_pred, max_depth=max_depth)
    fused_c = clip_prediction(fused, max_depth=max_depth)
    b = compute_depth_metrics(base_c, gt, max_depth=max_depth)
    f = compute_depth_metrics(fused_c, gt, max_depth=max_depth)
    if b is None or f is None:
        return None
    union = union_masks(masks, gt.shape[:2])
    return {
        "base": b,
        "fused": f,
        "inside_base": metrics_in_region(base_c, gt, union, max_depth),
        "inside_fused": metrics_in_region(fused_c, gt, union, max_depth),
        "outside_base": metrics_in_region(base_c, gt, ~union, max_depth),
        "outside_fused": metrics_in_region(fused_c, gt, ~union, max_depth),
        "coverage": float(union.mean()),
        "fraction_changed": float((fused != base_pred).mean()),
        "n_masks": len(masks),
    }

# =========================================================================
# Resumable inference over all samples (the slow part -- run once)
# =========================================================================
def _free_gpu():
    import gc
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


def load_scored(jsonl_path: str) -> Dict[int, Dict[str, Any]]:
    """Loads already-scored per-sample rows keyed by sample index."""
    done: Dict[int, Dict[str, Any]] = {}
    if os.path.exists(jsonl_path):
        with open(jsonl_path) as fh:
            for line in fh:
                line = line.strip()
                if not line:
                    continue
                try:
                    r = json.loads(line)
                    done[int(r["idx"])] = r
                except (json.JSONDecodeError, KeyError):
                    continue
    return done


def run_inference(num_samples: int, split: str, baseline_slug: str, sam2_size: str,
                  output_dir: str, device: Optional[str], max_depth: float,
                  sam2_quick_test: bool, checkpoints_dir: str,
                  relative: bool = False) -> str:
    """Scores baseline vs fused for every sample, appending to a JSONL. Skips
    indices already present, so re-running resumes where a dead session left off.

    relative=True (auto-enabled for a RELATIVE_SLUGS baseline): the checkpoint
    emits disparity, so each prediction is aligned to metric depth via a
    per-image scale+shift fit against GT (align_relative_depth) BEFORE fusion and
    scoring. Object-median fusion then runs on the aligned METRIC depth -- i.e.
    each object's pixels take the median of that object's metric depth, exactly
    as in the metric notebooks. base and fused are built from the SAME aligned
    prediction (one (s,t) per image), so the only difference between them stays
    the segmentation layer."""
    os.makedirs(output_dir, exist_ok=True)
    jsonl = os.path.join(output_dir, "kfold_persample.jsonl")
    done = load_scored(jsonl)
    print(f"{len(done)}/{num_samples} samples already scored (resuming).")
    if len(done) >= num_samples:
        print("Nothing to do -- all requested samples already scored.")
        return jsonl

    if device is None:
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
    # A preset slug (da2-metric-* / da2-relative-*) resolves to its HF id;
    # anything else is passed through as-is, so baseline_slug can be a HuggingFace
    # id OR a local path to a fine-tuned metric checkpoint (e.g. the metric MiDaS
    # from finetune_midas.py saved via save_pretrained).
    model_id = BASELINE_IDS.get(baseline_slug, baseline_slug)
    relative = relative or (baseline_slug in RELATIVE_SLUGS)
    print(f"device={device}  baseline={model_id}  sam2={sam2_size}"
          + ("  (relative -> scale+shift aligned)" if relative else "")
          + ("  (quick grid)" if sam2_quick_test else ""))

    depth_pipe = load_metric_depth_model(model_id, device)
    sam2_ckpt = ensure_sam2_checkpoint(sam2_size, checkpoints_dir)
    sam2_cfg = SAM2_PRESETS[sam2_size]["config"]
    mask_gen = load_sam2_mask_generator(sam2_ckpt, sam2_cfg, device, quick_test=sam2_quick_test)

    n_new = n_skip = 0
    with open(jsonl, "a") as out:
        for idx, item in iter_nyu(split, num_samples):
            if idx in done:
                continue
            try:
                image = np.array(item["image"].convert("RGB"))
                gt = np.array(item["depth_map"]).astype(np.float32)
                base_pred = predict_depth_meters(depth_pipe, image)
                if relative:
                    # disparity -> metric depth via per-image scale+shift on GT
                    base_pred, _s, _t = align_relative_depth(base_pred, gt, max_depth=max_depth)
                masks = generate_masks(mask_gen, image)
                row = score_sample(base_pred, gt, masks, max_depth)
            except Exception as e:
                n_skip += 1
                print(f"  [idx {idx}] skipped: {e}")
                continue
            if row is None:
                n_skip += 1
                continue
            row["idx"] = idx
            out.write(json.dumps(row) + "\n")
            out.flush()
            n_new += 1
            if n_new % 10 == 0 or (len(done) + n_new + n_skip) >= num_samples:
                print(f"  scored {len(done) + n_new}/{num_samples}  (skipped {n_skip})")
            if n_new % 50 == 0:
                _free_gpu()

    print(f"Inference done. {n_new} new samples scored, {n_skip} skipped.")
    _free_gpu()
    return jsonl

# =========================================================================
# 5-fold aggregation (pure post-processing over the cached scores)
# =========================================================================
def assign_folds(indices: List[int], n_folds: int, seed: int = 42) -> Dict[int, int]:
    """Deterministic disjoint folds: seeded shuffle of the sample indices,
    split into n_folds balanced chunks. Independent of the order samples were
    scored in, so resumed runs give the same folds."""
    idx = sorted(indices)
    perm = np.random.default_rng(seed).permutation(len(idx))
    fold_of: Dict[int, int] = {}
    for f, chunk in enumerate(np.array_split(perm, n_folds)):
        for pos in chunk:
            fold_of[idx[int(pos)]] = f
    return fold_of


def mean_metrics(rows: List[Dict[str, Any]], key: str) -> Dict[str, Optional[float]]:
    """Mean of each metric field over rows[*][key] (None entries skipped)."""
    acc: Dict[str, List[float]] = {f: [] for f in METRIC_FIELDS}
    for r in rows:
        m = r.get(key)
        if m is None:
            continue
        for f in METRIC_FIELDS:
            acc[f].append(m[f])
    return {f: (float(np.mean(acc[f])) if acc[f] else None) for f in METRIC_FIELDS}


def aggregate(jsonl_path: str, n_folds: int, seed: int, output_dir: str,
              baseline_slug: str, relative: bool = False) -> Dict[str, Any]:
    rows = list(load_scored(jsonl_path).values())
    if not rows:
        raise RuntimeError("No scored samples found -- run inference first.")
    fold_of = assign_folds([r["idx"] for r in rows], n_folds, seed)
    for r in rows:
        r["fold"] = fold_of[r["idx"]]

    # Per-fold means (each sample scored once, belongs to one fold).
    per_fold: Dict[int, Any] = {}
    for f in range(n_folds):
        fr = [r for r in rows if r["fold"] == f]
        per_fold[f] = {"n": len(fr), "base": mean_metrics(fr, "base"),
                       "fused": mean_metrics(fr, "fused")}

    # Across-fold mean/std (the headline CV numbers) + whole-dataset paired
    # improved/worsened/unchanged counts.
    summary: Dict[str, Any] = {}
    for field in METRIC_FIELDS:
        base_fold = [per_fold[f]["base"][field] for f in range(n_folds)
                     if per_fold[f]["base"][field] is not None]
        fused_fold = [per_fold[f]["fused"][field] for f in range(n_folds)
                      if per_fold[f]["fused"][field] is not None]
        diff_fold = [fu - b for b, fu in zip(base_fold, fused_fold)]
        d = [r["fused"][field] - r["base"][field] for r in rows]
        if field in LOWER_BETTER:
            improved = sum(x < -1e-9 for x in d)
            worsened = sum(x > 1e-9 for x in d)
        else:
            improved = sum(x > 1e-9 for x in d)
            worsened = sum(x < -1e-9 for x in d)
        summary[field] = {
            "base_mean": float(np.mean(base_fold)), "base_std": float(np.std(base_fold)),
            "fused_mean": float(np.mean(fused_fold)), "fused_std": float(np.std(fused_fold)),
            "diff_mean": float(np.mean(diff_fold)), "diff_std": float(np.std(diff_fold)),
            "improved": improved, "worsened": worsened, "unchanged": len(d) - improved - worsened,
        }

    region = {reg: {m: mean_metrics(rows, f"{reg}_{m}") for m in ("base", "fused")}
              for reg in ("inside", "outside")}
    pixel = {"mean_coverage": float(np.mean([r["coverage"] for r in rows])),
             "mean_fraction_changed": float(np.mean([r["fraction_changed"] for r in rows]))}

    result = {"n_samples": len(rows), "n_folds": n_folds, "seed": seed,
              "baseline_slug": baseline_slug, "relative": bool(relative),
              "fusion": "object-median",
              "per_fold": per_fold, "summary": summary, "region": region, "pixel_stats": pixel}
    with open(os.path.join(output_dir, "kfold_summary.json"), "w") as fh:
        json.dump(result, fh, indent=2)
    return result

# =========================================================================
# CSV + plots
# =========================================================================
def write_csvs(result: Dict[str, Any], output_dir: str):
    p1 = os.path.join(output_dir, "kfold_per_fold.csv")
    with open(p1, "w", newline="") as fh:
        fh.write("fold,n,model," + ",".join(METRIC_FIELDS) + "\n")
        for f in range(result["n_folds"]):
            for model in ("base", "fused"):
                m = result["per_fold"][f][model]
                vals = ",".join("" if m[k] is None else f"{m[k]:.6f}" for k in METRIC_FIELDS)
                fh.write(f"{f},{result['per_fold'][f]['n']},{model},{vals}\n")

    p2 = os.path.join(output_dir, "kfold_summary.csv")
    with open(p2, "w", newline="") as fh:
        fh.write("metric,base_mean,base_std,fused_mean,fused_std,"
                 "diff_mean,diff_std,improved,worsened,unchanged\n")
        for field in METRIC_FIELDS:
            s = result["summary"][field]
            fh.write(f"{field},{s['base_mean']:.6f},{s['base_std']:.6f},"
                     f"{s['fused_mean']:.6f},{s['fused_std']:.6f},"
                     f"{s['diff_mean']:.6f},{s['diff_std']:.6f},"
                     f"{s['improved']},{s['worsened']},{s['unchanged']}\n")
    print(f"Saved {p1}\nSaved {p2}")


def plot_summary(result: Dict[str, Any], output_dir: str, baseline_name: str, fused_name: str):
    import matplotlib.pyplot as plt
    c_b, c_f = plt.get_cmap("tab10")(0), plt.get_cmap("tab10")(1)
    fig, axes = plt.subplots(2, 4, figsize=(22, 10))
    axes = axes.ravel()
    for ax, field in zip(axes, METRIC_FIELDS):
        s = result["summary"][field]
        bars = ax.bar([0, 1], [s["base_mean"], s["fused_mean"]],
                      yerr=[s["base_std"], s["fused_std"]], capsize=6, color=[c_b, c_f])
        for x, v in zip([0, 1], [s["base_mean"], s["fused_mean"]]):
            ax.text(x, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
        lower = field in LOWER_BETTER
        ax.set_title(f"{field}  ({'lower is better' if lower else 'higher is better'})", fontsize=11)
        ax.set_xticks([0, 1])
        ax.set_xticklabels([baseline_name, fused_name], fontsize=8)
        ax.spines[["top", "right"]].set_visible(False)

    axes[7].axis("off")
    ar, d1 = result["summary"]["abs_rel"], result["summary"]["delta1"]
    axes[7].text(
        0.02, 0.95,
        f"{result['n_folds']}-fold CV over {result['n_samples']} images\n"
        f"error bars = std across folds\n\n"
        f"AbsRel: {ar['base_mean']:.4f} -> {ar['fused_mean']:.4f}\n"
        f"  diff {ar['diff_mean']:+.4f} +/- {ar['diff_std']:.4f}\n"
        f"  improved {ar['improved']}, worsened {ar['worsened']}, unchanged {ar['unchanged']}\n\n"
        f"delta1: {d1['base_mean']:.4f} -> {d1['fused_mean']:.4f}\n"
        f"  diff {d1['diff_mean']:+.4f} +/- {d1['diff_std']:.4f}\n\n"
        f"mask coverage {result['pixel_stats']['mean_coverage']*100:.1f}% of pixels\n"
        f"pixels changed by fusion {result['pixel_stats']['mean_fraction_changed']*100:.1f}%",
        transform=axes[7].transAxes, va="top", fontsize=10)

    fig.suptitle(f"Depth-only vs SAM2+depth (object-median fusion) -- "
                 f"{result['n_folds']}-fold CV, same backbone", fontsize=15)
    plt.tight_layout()
    p = os.path.join(output_dir, "kfold_comparison.png")
    fig.savefig(p, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {p}")


def plot_per_fold(result: Dict[str, Any], output_dir: str):
    import matplotlib.pyplot as plt
    nf = result["n_folds"]
    x = np.arange(nf)
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, field in zip(axes, ["abs_rel", "delta1"]):
        base = [result["per_fold"][f]["base"][field] for f in range(nf)]
        fused = [result["per_fold"][f]["fused"][field] for f in range(nf)]
        ax.plot(x, base, "o-", label="depth-only")
        ax.plot(x, fused, "s-", label="seg+depth (fused)")
        lower = field in LOWER_BETTER
        ax.set_title(f"{field} per fold  ({'lower is better' if lower else 'higher is better'})", fontsize=11)
        ax.set_xticks(x)
        ax.set_xticklabels([f"fold {i}" for i in range(nf)])
        ax.legend()
        ax.spines[["top", "right"]].set_visible(False)
    fig.suptitle("Per-fold stability of the seg-vs-depth comparison", fontsize=14)
    plt.tight_layout()
    p = os.path.join(output_dir, "kfold_per_fold.png")
    fig.savefig(p, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {p}")


def plot_region(result: Dict[str, Any], output_dir: str, baseline_name: str, fused_name: str):
    import matplotlib.pyplot as plt
    reg = result["region"]
    c_b, c_f = plt.get_cmap("tab10")(0), plt.get_cmap("tab10")(1)
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    for ax, field in zip(axes, ["abs_rel", "rmse", "delta1"]):
        vb = [reg["inside"]["base"][field], reg["outside"]["base"][field]]
        vf = [reg["inside"]["fused"][field], reg["outside"]["fused"][field]]
        vb = [0.0 if v is None else v for v in vb]
        vf = [0.0 if v is None else v for v in vf]
        x = np.arange(2)
        ax.bar(x - 0.18, vb, width=0.36, label=baseline_name, color=c_b)
        ax.bar(x + 0.18, vf, width=0.36, label=fused_name, color=c_f)
        lower = field in LOWER_BETTER
        ax.set_title(f"{field}  ({'lower is better' if lower else 'higher is better'})", fontsize=11)
        ax.set_xticks(x)
        ax.set_xticklabels(["inside masks", "outside masks"], fontsize=9)
        ax.legend(fontsize=8)
        ax.spines[["top", "right"]].set_visible(False)
    fig.suptitle("Where fusion acts: error inside vs outside SAM2 masks "
                 "(outside bars should nearly match -- sanity check)", fontsize=13)
    plt.tight_layout()
    p = os.path.join(output_dir, "kfold_region.png")
    fig.savefig(p, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {p}")

# =========================================================================
# Orchestration
# =========================================================================
def print_verdict(result: Dict[str, Any], baseline_name: str):
    ar = result["summary"]["abs_rel"]
    d1 = result["summary"]["delta1"]
    print(f"\n--- {result['n_folds']}-fold verdict over {result['n_samples']} images "
          f"(vs {baseline_name}) ---")
    print(f"AbsRel: {ar['base_mean']:.4f}+/-{ar['base_std']:.4f} -> "
          f"{ar['fused_mean']:.4f}+/-{ar['fused_std']:.4f}  "
          f"(diff {ar['diff_mean']:+.4f}+/-{ar['diff_std']:.4f}; "
          f"improved {ar['improved']}, worsened {ar['worsened']}, unchanged {ar['unchanged']})")
    print(f"delta1: {d1['base_mean']:.4f}+/-{d1['base_std']:.4f} -> "
          f"{d1['fused_mean']:.4f}+/-{d1['fused_std']:.4f}  "
          f"(diff {d1['diff_mean']:+.4f}+/-{d1['diff_std']:.4f})")


def run_kfold(num_samples: int = 3000, n_folds: int = 5, split: str = "train",
              baseline_slug: str = "da2-metric-small", sam2_size: str = "small",
              output_dir: str = "seg_depth_kfold_out", device: Optional[str] = None,
              max_depth: float = 10.0, seed: int = 42, sam2_quick_test: bool = False,
              checkpoints_dir: str = "checkpoints", relative: bool = False) -> Dict[str, Any]:
    """End-to-end: resumable inference -> 5-fold aggregation -> CSVs + plots.
    Safe to call repeatedly; inference resumes and aggregation is recomputed.

    relative=True (auto for a da2-relative-* slug) aligns the disparity output to
    metric depth per image before fusion/scoring (align_relative_depth)."""
    relative = relative or (baseline_slug in RELATIVE_SLUGS)
    jsonl = run_inference(num_samples, split, baseline_slug, sam2_size, output_dir,
                          device, max_depth, sam2_quick_test, checkpoints_dir,
                          relative=relative)
    result = aggregate(jsonl, n_folds, seed, output_dir, baseline_slug, relative=relative)
    write_csvs(result, output_dir)
    tag = " (aligned)" if relative else ""
    fused_name = f"SAM2 + {baseline_slug}{tag}"
    plot_summary(result, output_dir, f"{baseline_slug}{tag}", fused_name)
    plot_per_fold(result, output_dir)
    plot_region(result, output_dir, f"{baseline_slug}{tag}", fused_name)
    print_verdict(result, baseline_slug)
    return result


def main():
    ap = argparse.ArgumentParser(
        description="5-fold CV of SAM2+depth (object-median) vs depth-only on NYU Depth V2")
    ap.add_argument("--split", default="train", help="'train' (default; validation too small for 3000) or 'validation'")
    ap.add_argument("--num-samples", type=int, default=3000)
    ap.add_argument("--n-folds", type=int, default=5)
    ap.add_argument("--baseline", default="da2-metric-small",
                    help="preset slug (%s) OR a HuggingFace id / local path to a "
                         "metric checkpoint (e.g. a fine-tuned metric MiDaS dir)"
                         % " | ".join(BASELINE_IDS))
    ap.add_argument("--relative", action="store_true",
                    help="baseline emits relative disparity -> per-image scale+shift "
                         "align to GT before scoring (auto-enabled for da2-relative-* slugs)")
    ap.add_argument("--sam2-size", default="small", choices=list(SAM2_PRESETS))
    ap.add_argument("--output-dir", default="seg_depth_kfold_out")
    ap.add_argument("--checkpoints-dir", default="checkpoints")
    ap.add_argument("--device", default=None, help="cuda | cpu (auto-detected if omitted)")
    ap.add_argument("--max-depth", type=float, default=10.0)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--sam2-quick-test", action="store_true", help="Sparser SAM2 grid (faster, coarser)")
    a = ap.parse_args()
    run_kfold(num_samples=a.num_samples, n_folds=a.n_folds, split=a.split,
              baseline_slug=a.baseline, sam2_size=a.sam2_size, output_dir=a.output_dir,
              device=a.device, max_depth=a.max_depth, seed=a.seed,
              sam2_quick_test=a.sam2_quick_test, checkpoints_dir=a.checkpoints_dir,
              relative=a.relative)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile seg_cascade_kfold.py
#!/usr/bin/env python3
"""
seg_cascade_kfold.py

5-FOLD CROSS-VALIDATION of a SEGMENTATION-CASCADE depth model
(SAM2 gates the depth model's INPUT) against BOTH the same depth model with no
segmentation AND a mask-matched control, on NYU Depth V2.

WHAT THIS IS
------------
seg_depth_kfold.py evaluates LATE fusion: Depth Anything V2 runs on the original
image, then SAM2 masks post-process the depth map (each object's pixels take that
object's median depth). There the two models are independent.

This module evaluates a CASCADE instead -- segmentation is a stage the image
passes THROUGH before depth estimation:

  base  : image                        -> DA V2 -> depth      (depth-only)
  casc  : image with ~union set gray   -> DA V2 -> depth      (SAM2 gate)
  ctrl  : image with ~roll(union) gray -> DA V2 -> depth      (control)

WHY THE MASK IS BAKED INTO THE PIXELS
-------------------------------------
Depth Anything V2 is a monocular RGB estimator: predict_depth_meters() accepts an
image and nothing else (no mask input, no prompt). A mask cannot be handed to it.
The only training-free way to make segmentation "feed" the depth model is to
change the pixels the model sees -- here, everything outside the SAM2 region is
replaced with neutral gray. The depth model still only ever sees RGB.

WHY THE CONTROL ARM EXISTS (read this before quoting any result)
----------------------------------------------------------------
Masking changes two things at once: (a) WHICH region is hidden, and (b) HOW MANY
pixels are hidden. A cascade that beats the depth-only baseline may be exploiting
(d) SAM2's semantics, or merely reacting to (b) a big flat gray patch -- DA V2
leans on global context, so any large occlusion perturbs it.

The control isolates those: `roll(union)` has EXACTLY the same area and shape as
the real union, differing only in WHERE it sits. So:

  casc vs base  -> total effect of the cascade
  ctrl vs base  -> effect of masking that much area, with no semantics
  casc vs ctrl  -> the part attributable to SEGMENTATION  <-- the real question

If casc ~= ctrl, the "cascade" is a masking artefact, not segmentation. The
per-sample control IoU is recorded and plotted so you can see when the control is
weak (it necessarily degrades as union coverage approaches 1, because a rolled
copy of a near-full-frame mask overlaps its original).

COVERAGE IS THE MAIN RISK
-------------------------
SAM2's automatic generator proposes a mask for essentially every surface, so
union can cover 85-95% of the frame. At that point casc ~= base (nothing is
masked out) and the experiment answers nothing. Run --num-samples 20 first and
READ mean coverage before committing to 3000 images; if it is above ~0.8, tighten
--min-mask-area / --max-coverage and re-smoke. Coverage is written to every row
and printed before any verdict.

RELATIVE BACKBONES
------------------
A relative checkpoint emits disparity, so each of the three predictions needs its
own per-image scale+shift fit onto GT inverse depth (the MiDaS / DA-V2 protocol,
align_relative_depth). Note this means part of any difference is the fit
RE-ADAPTING to the masked image rather than a change in depth quality. Use
--align-region inside to fit only on the pixels we actually score (closer to the
claim, but no longer comparable to the metric com1 notebooks); the default is
"all", the existing protocol.

USAGE
-----
python seg_cascade_kfold.py --selftest                    # CPU, no models, no download
python seg_cascade_kfold.py --num-samples 20 --sam2-quick-test
python seg_cascade_kfold.py --num-samples 3000 --n-folds 5 --baseline da2-metric-small

Resumable: per-sample scores are appended to <output-dir>/cascade_per_sample.jsonl
as each image finishes; re-running skips images already scored. The control shift
is seeded per sample index (not from a running RNG), so a resumed run reproduces
the identical control for every image.

Reuses seg_depth_kfold.py (imported, not copied) for the metrics, depth loader,
SAM2 generator and fold assignment, so numbers stay directly comparable with the
com1 notebooks.
"""

import argparse
import json
import os
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import cv2

import seg_depth_kfold as K

# The three ablation arms, in report order. Index 0 is the reference arm that all
# paired differences are taken against.
ARMS = ("base", "casc", "ctrl")
REF_ARM = "base"
# The comparison that actually answers the thesis question.
KEY_PAIR = ("casc", "ctrl")

# Neutral gray used to fill the masked-out region. 128 is the midpoint of the
# uint8 range and close to the ImageNet mean the backbone was normalised with.
FILL_VALUE = 128

# Regions the three arms are scored over:
#   inside  -- the SAM2 union (where the cascade keeps real pixels)
#   outside -- its complement (where the cascade sees only flat gray)
#   inner   -- the union eroded by INNER_ERODE_FRAC (boundary-bleed check)
INNER_ERODE_FRAC = 0.02

# The dataset and its split ordering. Defined here (rather than read off
# seg_depth_kfold, which does not export it) because iter_nyu below reads the
# split directly -- it must, in order to skip the fine-tune prefix.
NYU_DATASET = "sayakpaul/nyu_depth_v2"


# =========================================================================
# Mask -> region (pure numpy, no models -- exercised by --selftest)
# =========================================================================
def select_masks(mask_dicts: List[Dict[str, Any]], shape: Tuple[int, int],
                 min_mask_area: float = 0.0,
                 max_coverage: float = 1.0) -> Tuple[np.ndarray, int]:
    """Boolean union of the SAM2 masks, after two optional filters.

    min_mask_area -- drop masks smaller than this FRACTION of the image (a
                     fraction rather than a pixel count so it transfers across
                     resolutions). 0.0 keeps everything.
    max_coverage  -- cap the union at this fraction of the image. Masks are added
                     SMALLEST FIRST and we stop at the first one that would push
                     the union over the cap, so the largest masks (walls, floor,
                     ceiling -- usually the background) are the ones dropped.
                     1.0 disables the cap.

    Returns (union, n_kept). With the defaults this is exactly
    seg_depth_kfold.union_masks() -- the knobs exist because high coverage can
    make the whole ablation degenerate (see module docstring).
    """
    h, w = shape
    total = float(h * w)
    min_px = min_mask_area * total

    # K._fit_mask resizes a mask to (h, w); reusing it keeps the geometry
    # identical to the late-fusion code path.
    sized = [(float(m["area"]), K._fit_mask(m["segmentation"].astype(bool), shape))
             for m in mask_dicts if float(m["area"]) >= min_px]
    sized.sort(key=lambda t: t[0])           # smallest first

    union = np.zeros(shape, dtype=bool)
    kept = 0
    for _area, m in sized:
        candidate = union | m
        if candidate.mean() > max_coverage:
            break                            # this mask and all larger ones go
        union = candidate
        kept += 1
    return union, kept


def apply_region_fill(image_rgb: np.ndarray, region: np.ndarray,
                      fill: int = FILL_VALUE) -> np.ndarray:
    """Copy of the image with `region` (True = fill) set to a flat `fill` value.

    This is the cascade's entire mechanism: it is what "the segmentation output
    goes into the depth model" means in practice.
    """
    out = image_rgb.copy()
    out[region] = fill
    return out


def control_region(union: np.ndarray, rng: np.random.Generator,
                   n_candidates: int = 8,
                   mode: str = "roll") -> Tuple[np.ndarray, float]:
    """A mask-matched, semantically-empty stand-in for `union`.

    mode="roll" circularly shifts the union by a random (dy, dx), which preserves
    its area and shape EXACTLY and changes only its location. Among
    `n_candidates` random shifts we keep the one with the LOWEST IoU against the
    union -- a shift of (0,0) would otherwise occasionally return the union
    itself and silently turn the control into a copy of the treatment.

    Returns (region, iou_with_union). iou is the diagnostic that says whether the
    control was meaningful for this image: near 0 is a clean contrast, near 1
    means the two arms saw nearly the same pixels and the comparison is void.
    """
    h, w = union.shape
    if mode != "roll":
        raise ValueError(f"unknown control mode {mode!r}")

    best, best_iou = None, None
    for _ in range(max(1, n_candidates)):
        dy, dx = int(rng.integers(0, h)), int(rng.integers(0, w))
        region = np.roll(np.roll(union, dy, axis=0), dx, axis=1)
        inter = int(np.logical_and(region, union).sum())
        uni = int(np.logical_or(region, union).sum())
        iou = float(inter / uni) if uni else 0.0
        if best_iou is None or iou < best_iou:
            best, best_iou = region, iou
    return best, best_iou


def erode_region(region: np.ndarray, frac: float = INNER_ERODE_FRAC) -> np.ndarray:
    """`region` shrunk by a kernel of frac*min(H,W).

    Scoring the eroded interior shows whether an effect lives in the object or
    only at the gray/object boundary, where the fill is most out of distribution
    for the backbone.
    """
    h, w = region.shape
    k = max(1, int(round(frac * min(h, w))))
    if k <= 1:
        return region.copy()
    kernel = np.ones((k, k), np.uint8)
    return cv2.erode(region.astype(np.uint8), kernel, iterations=1).astype(bool)


# =========================================================================
# Scoring one image across all three arms
# =========================================================================
def score_sample_three(base_pred: np.ndarray, casc_pred: np.ndarray,
                       ctrl_pred: np.ndarray, gt: np.ndarray, union: np.ndarray,
                       ctrl_iou: float, n_masks: int,
                       max_depth: float) -> Optional[Dict[str, Any]]:
    """Whole-image + inside/inner/outside metrics for base, casc and ctrl.

    Returns None if ANY arm fails to score, so every row describes the same
    image set and the paired improved/worsened counts stay honest.
    """
    preds = {"base": base_pred, "casc": casc_pred, "ctrl": ctrl_pred}
    clipped = {a: K.clip_prediction(p, max_depth=max_depth) for a, p in preds.items()}

    row: Dict[str, Any] = {}
    for arm, p in clipped.items():
        m = K.compute_depth_metrics(p, gt, max_depth=max_depth)
        if m is None:
            return None
        row[arm] = m

    regions = {"inside": union, "outside": ~union, "inner": erode_region(union)}
    for rname, rmask in regions.items():
        for arm, p in clipped.items():
            row[f"{rname}_{arm}"] = K.metrics_in_region(p, gt, rmask, max_depth)

    row["coverage"] = float(union.mean())
    row["ctrl_iou"] = float(ctrl_iou)
    row["n_masks"] = int(n_masks)
    return row


def _align_arm(pred: np.ndarray, gt: np.ndarray, region: np.ndarray,
               align_region: str, max_depth: float) -> np.ndarray:
    """Per-image scale+shift alignment for a relative (disparity) backbone.

    align_region="all"    -- fit on every valid pixel (the existing protocol).
    align_region="inside" -- fit only on pixels inside `region`, i.e. the ones we
                             actually score, so the fit is not dominated by the
                             flat-gray area the mask created.

    align_relative_depth() couples the fit and the transform, so for "inside" we
    call it on region-restricted copies purely to recover (s, t), then apply that
    transform to the FULL prediction.
    """
    if align_region == "all":
        return K.align_relative_depth(pred, gt, max_depth=max_depth)[0]
    if align_region != "inside":
        raise ValueError(f"unknown align_region {align_region!r}")

    gt_r = np.where(region, gt, 0.0).astype(np.float32)
    pred_r = np.where(region, pred, np.nan).astype(np.float32)
    _aligned, s, t = K.align_relative_depth(pred_r, gt_r, max_depth=max_depth)
    return (1.0 / np.clip(s * pred.astype(np.float64) + t, 1e-6, None)).astype(np.float32)


# =========================================================================
# Resumable inference over all samples (the slow part -- run once)
# =========================================================================
def run_inference(num_samples: int, split: str, baseline_slug: str, sam2_size: str,
                  output_dir: str, device: Optional[str], max_depth: float,
                  sam2_quick_test: bool, checkpoints_dir: str,
                  relative: bool = False, align_region: str = "all",
                  min_mask_area: float = 0.0, max_coverage: float = 1.0,
                  seed: int = 42, eval_offset: int = 0) -> str:
    """Scores all three arms for every sample, appending to a JSONL. Skips
    indices already present, so re-running resumes where a dead session left off.

    Per image: SAM2 masks -> union (filtered) -> rolled control -> three depth
    forwards (original, union-filled, control-filled) -> aligned if needed ->
    score_sample_three. SAM2 runs ONCE per image; the depth model runs three
    times (up from once in the late-fusion module).

    eval_offset -- skip the first `eval_offset` items of the split, so the scored
    images are disjoint from the prefix a fine-tuned checkpoint trained on (see
    finetune_backbone.py). Indices stored in the JSONL are ABSOLUTE, so a
    checkpoint trained on [0, eval_offset) can never appear here.
    """
    os.makedirs(output_dir, exist_ok=True)
    jsonl = os.path.join(output_dir, "cascade_per_sample.jsonl")
    done = K.load_scored(jsonl)
    if done and eval_offset:
        # Scores cached from a run with a different offset describe a different
        # image set under the same indices -- resuming would silently mix them.
        seen = {int(r.get("eval_offset", 0)) for r in done.values()}
        if seen != {eval_offset}:
            raise ValueError(
                f"{jsonl} holds scores written with eval_offset={sorted(seen)} but "
                f"this run uses {eval_offset}. The cached rows are a DIFFERENT image "
                f"set under the same indices, so they cannot be resumed. Delete the "
                f"file to re-run from scratch (or restore the old --eval-offset).")
    print(f"{len(done)}/{num_samples} samples already scored (resuming).")
    if len(done) >= num_samples:
        print("Nothing to do -- all requested samples already scored.")
        return jsonl

    if device is None:
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
    model_id = K.BASELINE_IDS.get(baseline_slug, baseline_slug)
    relative = relative or (baseline_slug in K.RELATIVE_SLUGS)
    print(f"device={device}  baseline={model_id}  sam2={sam2_size}"
          + ("  (relative -> scale+shift aligned)" if relative else "")
          + (f"  align_region={align_region}" if relative else "")
          + (f"  min_mask_area={min_mask_area}  max_coverage={max_coverage}"
             if (min_mask_area > 0 or max_coverage < 1.0) else "")
          + (f"  {split}[{eval_offset}:{eval_offset + num_samples}]" if eval_offset
             else f"  {split}[0:{num_samples}]")
          + ("  (quick grid)" if sam2_quick_test else ""))

    depth_pipe = K.load_metric_depth_model(model_id, device)
    sam2_ckpt = K.ensure_sam2_checkpoint(sam2_size, checkpoints_dir)
    sam2_cfg = K.SAM2_PRESETS[sam2_size]["config"]
    mask_gen = K.load_sam2_mask_generator(sam2_ckpt, sam2_cfg, device,
                                          quick_test=sam2_quick_test)

    n_new = n_skip = n_seen = 0
    with open(jsonl, "a") as out:
        for idx, item in iter_nyu(split, num_samples, eval_offset):
            n_seen += 1
            if idx in done:
                continue
            try:
                image = np.array(item["image"].convert("RGB"))
                gt = np.array(item["depth_map"]).astype(np.float32)
                shape = image.shape[:2]

                masks = K.generate_masks(mask_gen, image)
                union, n_kept = select_masks(masks, shape, min_mask_area, max_coverage)
                if not union.any():
                    raise ValueError("empty mask union -- nothing to gate on")

                # Seeded per INDEX, not from a running RNG, so resuming a dead
                # session reproduces the same control for every image.
                rng = np.random.default_rng([seed, idx])
                ctrl, ctrl_iou = control_region(union, rng)

                base_pred = K.predict_depth_meters(depth_pipe, image)
                casc_pred = K.predict_depth_meters(
                    depth_pipe, apply_region_fill(image, ~union))
                ctrl_pred = K.predict_depth_meters(
                    depth_pipe, apply_region_fill(image, ~ctrl))

                if relative:
                    base_pred = _align_arm(base_pred, gt, union, align_region, max_depth)
                    casc_pred = _align_arm(casc_pred, gt, union, align_region, max_depth)
                    ctrl_pred = _align_arm(ctrl_pred, gt, union, align_region, max_depth)

                row = score_sample_three(base_pred, casc_pred, ctrl_pred, gt, union,
                                         ctrl_iou, n_kept, max_depth)
            except Exception as e:
                n_skip += 1
                print(f"  [idx {idx}] skipped: {e}")
                continue
            if row is None:
                n_skip += 1
                continue
            row["idx"] = idx
            row["eval_offset"] = eval_offset   # pins the row to its image set
            out.write(json.dumps(row) + "\n")
            out.flush()
            n_new += 1
            if n_new % 10 == 0 or (len(done) + n_new + n_skip) >= num_samples:
                print(f"  scored {len(done) + n_new}/{num_samples}  (skipped {n_skip})")
            if n_new % 50 == 0:
                K._free_gpu()

    print(f"Inference done. {n_new} new samples scored, {n_skip} skipped.")
    # The stream ended before supplying the requested samples -- the split is
    # smaller than the requested [eval_offset, eval_offset + num_samples). Say so
    # loudly: the CSVs would otherwise look like a clean result on fewer images.
    if n_seen < num_samples:
        raise ValueError(
            f"{split} ran out after {n_seen} samples, but {num_samples} were "
            f"requested from index {eval_offset}. The split holds only "
            f"{eval_offset + n_seen} usable images here -- see probe_split_size(). "
            f"Lower --num-samples, or lower --eval-offset if no fine-tune prefix "
            f"needs to stay excluded. The {n_new + len(done)} already-scored rows "
            f"are kept in {jsonl}.")
    K._free_gpu()
    return jsonl


def iter_nyu(split: str, num_samples: int, offset: int = 0):
    """Yields `(absolute_index, item)` for the samples at positions
    `[offset, offset + num_samples)` of `split`.

    This module does NOT use K.iter_nyu, because that one always starts at 0 and
    renumbers from 0. Here the index is the ABSOLUTE position in the split, which
    is what makes the fine-tune/evaluation split disjoint: fine-tuning consumes
    [0, offset), this yields [offset, offset + num_samples), and the per-sample
    JSONL keys on the absolute index so a resumed run and the fold assignment
    both stay stable.

    The loader is duplicated from seg_depth_kfold (rather than imported) so the
    offset can be applied to the stream position -- and it keeps the same
    datasets>=4 parquet fallback, because the fine-tune script reads the split
    through that identical path. If the two orderings ever diverged, `offset`
    would stop guaranteeing disjointness.
    """
    from datasets import load_dataset
    try:
        ds = load_dataset(NYU_DATASET, split=split, streaming=True)
    except RuntimeError:
        data_files = {split: f"hf://datasets/{NYU_DATASET}@refs/convert/parquet/"
                              f"default/partial-{split}/*.parquet"}
        ds = load_dataset("parquet", data_files=data_files, split=split, streaming=True)
    for i, item in enumerate(ds):
        if i < offset:
            continue
        if i >= offset + num_samples:
            break
        yield i, item


# =========================================================================
# 5-fold aggregation (pure post-processing over the cached scores)
# =========================================================================
def _pair_counts(diffs: List[float], field: str) -> Tuple[int, int, int]:
    """(improved, worsened, unchanged) for a list of per-image differences.

    Direction depends on the metric: lower is better for abs_rel/rmse/..., higher
    for delta1/2/3 (LOWER_BETTER in seg_depth_kfold).
    """
    if field in K.LOWER_BETTER:
        improved = sum(d < -1e-9 for d in diffs)
        worsened = sum(d > 1e-9 for d in diffs)
    else:
        improved = sum(d > 1e-9 for d in diffs)
        worsened = sum(d < -1e-9 for d in diffs)
    return improved, worsened, len(diffs) - improved - worsened


def aggregate(jsonl_path: str, n_folds: int, seed: int, output_dir: str,
              baseline_slug: str, relative: bool = False,
              align_region: str = "all") -> Dict[str, Any]:
    rows = list(K.load_scored(jsonl_path).values())
    if not rows:
        raise RuntimeError("No scored samples found -- run inference first.")
    fold_of = K.assign_folds([r["idx"] for r in rows], n_folds, seed)
    for r in rows:
        r["fold"] = fold_of[r["idx"]]

    per_fold: Dict[int, Any] = {}
    for f in range(n_folds):
        fr = [r for r in rows if r["fold"] == f]
        per_fold[f] = {"n": len(fr)}
        for arm in ARMS:
            per_fold[f][arm] = K.mean_metrics(fr, arm)

    summary: Dict[str, Any] = {}
    for field in K.METRIC_FIELDS:
        fold_means = {arm: [per_fold[f][arm][field] for f in range(n_folds)
                            if per_fold[f][arm][field] is not None] for arm in ARMS}
        img = {arm: [r[arm][field] for r in rows] for arm in ARMS}

        # Paired per-image differences for each arm against the reference, plus
        # the headline casc-vs-ctrl contrast.
        pairs = {"casc_vs_base": ("casc", "base"), "ctrl_vs_base": ("ctrl", "base"),
                 "casc_vs_ctrl": ("casc", "ctrl")}
        entry: Dict[str, Any] = {}
        for arm in ARMS:
            entry[f"{arm}_mean"] = float(np.mean(fold_means[arm]))
            entry[f"{arm}_std"] = float(np.std(fold_means[arm]))
        for label, (a, b) in pairs.items():
            fold_diff = [x - y for x, y in zip(fold_means[a], fold_means[b])]
            img_diff = [x - y for x, y in zip(img[a], img[b])]
            entry[f"{label}_mean"] = float(np.mean(fold_diff))
            entry[f"{label}_std"] = float(np.std(fold_diff))
            imp, wor, unch = _pair_counts(img_diff, field)
            entry[f"{label}_improved"] = imp
            entry[f"{label}_worsened"] = wor
            entry[f"{label}_unchanged"] = unch
        summary[field] = entry

    region = {reg: {arm: K.mean_metrics(rows, f"{reg}_{arm}") for arm in ARMS}
              for reg in ("inside", "inner", "outside")}
    pixel = {"mean_coverage": float(np.mean([r["coverage"] for r in rows])),
             "mean_ctrl_iou": float(np.mean([r["ctrl_iou"] for r in rows])),
             "mean_n_masks": float(np.mean([r["n_masks"] for r in rows]))}

    # The offset is provenance, not a knob: without it the summary cannot say
    # which images produced these numbers, and a later fine-tune would have no
    # way to prove it stayed disjoint.
    offsets = {int(r.get("eval_offset", 0)) for r in rows}
    result = {"n_samples": len(rows), "n_folds": n_folds, "seed": seed,
              "baseline_slug": baseline_slug, "relative": bool(relative),
              "align_region": align_region, "mechanism": "masked-input cascade",
              "fill_value": FILL_VALUE, "arms": list(ARMS), "key_pair": list(KEY_PAIR),
              "eval_offset": sorted(offsets)[0] if len(offsets) == 1 else sorted(offsets),
              "per_fold": per_fold, "summary": summary, "region": region,
              "pixel_stats": pixel}
    with open(os.path.join(output_dir, "cascade_summary.json"), "w") as fh:
        json.dump(result, fh, indent=2)
    return result


# =========================================================================
# CSV + plots
# =========================================================================
def write_csvs(result: Dict[str, Any], output_dir: str):
    p1 = os.path.join(output_dir, "cascade_per_fold.csv")
    with open(p1, "w", newline="") as fh:
        fh.write("fold,n,arm," + ",".join(K.METRIC_FIELDS) + "\n")
        for f in range(result["n_folds"]):
            for arm in ARMS:
                m = result["per_fold"][f][arm]
                vals = ",".join("" if m[k] is None else f"{m[k]:.6f}" for k in K.METRIC_FIELDS)
                fh.write(f"{f},{result['per_fold'][f]['n']},{arm},{vals}\n")

    # The headline table: reference vs each arm vs the key contrast, per metric.
    p2 = os.path.join(output_dir, "cascade_summary.csv")
    cols = ["metric"]
    for arm in ARMS:
        cols += [f"{arm}_mean", f"{arm}_std"]
    for label in ("casc_vs_base", "ctrl_vs_base", "casc_vs_ctrl"):
        cols += [f"{label}_mean", f"{label}_std", f"{label}_improved",
                 f"{label}_worsened", f"{label}_unchanged"]
    counts = {f"{lbl}_{k}" for lbl in ("casc_vs_base", "ctrl_vs_base", "casc_vs_ctrl")
              for k in ("improved", "worsened", "unchanged")}
    with open(p2, "w", newline="") as fh:
        fh.write(",".join(cols) + "\n")
        for field in K.METRIC_FIELDS:
            s = result["summary"][field]
            vals = [(f"{s[c]:.6f}" if c not in counts else str(s[c])) for c in cols[1:]]
            fh.write(f"{field}," + ",".join(vals) + "\n")

    # Region breakdown (inside / inner / outside) -- the cascade's claim lives in
    # the "inside" row, so it gets its own table rather than living only in a plot.
    p3 = os.path.join(output_dir, "cascade_region.csv")
    with open(p3, "w", newline="") as fh:
        fh.write("region,arm," + ",".join(K.METRIC_FIELDS) + "\n")
        for reg in ("inside", "inner", "outside"):
            for arm in ARMS:
                m = result["region"][reg][arm]
                vals = ",".join("" if m[k] is None else f"{m[k]:.6f}" for k in K.METRIC_FIELDS)
                fh.write(f"{reg},{arm},{vals}\n")
    print(f"Saved {p1}\nSaved {p2}\nSaved {p3}")


def plot_three_arm(result: Dict[str, Any], output_dir: str, labels: Dict[str, str]):
    import matplotlib.pyplot as plt
    cmap = plt.get_cmap("tab10")
    colors = [cmap(0), cmap(1), cmap(3)]
    fig, axes = plt.subplots(2, 4, figsize=(22, 10))
    axes = axes.ravel()
    for ax, field in zip(axes, K.METRIC_FIELDS):
        s = result["summary"][field]
        means = [s[f"{a}_mean"] for a in ARMS]
        stds = [s[f"{a}_std"] for a in ARMS]
        bars = ax.bar([0, 1, 2], means, yerr=stds, capsize=6, color=colors)
        for bar, v in zip(bars, means):
            ax.text(bar.get_x() + bar.get_width() / 2, v, f"{v:.4f}",
                    ha="center", va="bottom", fontsize=9)
        ax.set_xticks([0, 1, 2])
        ax.set_xticklabels([labels[a] for a in ARMS], fontsize=9)
        ax.set_title(field, fontsize=11)
        ax.grid(axis="y", alpha=0.3)
    axes[-1].axis("off")
    fig.suptitle("Whole-image: depth-only vs segmentation cascade vs mask-matched control",
                 fontsize=14)
    fig.tight_layout()
    p = os.path.join(output_dir, "cascade_three_arm.png")
    fig.savefig(p, dpi=110, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {p}")


def plot_region_three(result: Dict[str, Any], output_dir: str, labels: Dict[str, str]):
    """The headline plot. The cascade only keeps real pixels INSIDE the union, so
    that is where its claim is; 'inner' guards against boundary artefacts and
    'outside' shows what masking cost."""
    import matplotlib.pyplot as plt
    cmap = plt.get_cmap("tab10")
    colors = [cmap(0), cmap(1), cmap(3)]
    regs = ["inside", "inner", "outside"]
    fields = ["abs_rel", "rmse", "delta1"]
    fig, axes = plt.subplots(1, 3, figsize=(19, 6))
    for ax, field in zip(axes, fields):
        x = np.arange(len(regs))
        width = 0.26
        for i, arm in enumerate(ARMS):
            vals = [result["region"][r][arm][field] for r in regs]
            ax.bar(x + (i - 1) * width, vals, width, label=labels[arm], color=colors[i])
        ax.set_xticks(x)
        ax.set_xticklabels(["inside union", "inner (eroded)", "outside union"], fontsize=9)
        ax.set_title(field, fontsize=11)
        ax.grid(axis="y", alpha=0.3)
    axes[0].legend(fontsize=9)
    fig.suptitle("Where the cascade acts: inside vs inner vs outside the SAM2 union "
                 "(lower better for abs_rel/rmse, higher for delta1)", fontsize=13)
    fig.tight_layout()
    p = os.path.join(output_dir, "cascade_region.png")
    fig.savefig(p, dpi=110, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {p}")


def plot_control_check(result: Dict[str, Any], output_dir: str, jsonl_path: str):
    """Is the control meaningful, and does the cascade effect survive it?

    Left: per-sample control IoU (near 0 = clean contrast, near 1 = the control
          was nearly a copy of the treatment).
    Right: cascade effect vs control effect, per image. Points ABOVE the diagonal
          are images where SAM2's semantics beat a matched-area mask of the same
          shape -- which is the only evidence that segmentation is doing work.
    """
    import matplotlib.pyplot as plt
    rows = list(K.load_scored(jsonl_path).values())
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    ious = [r["ctrl_iou"] for r in rows]
    axes[0].hist(ious, bins=30, color=plt.get_cmap("tab10")(3))
    axes[0].axvline(float(np.mean(ious)), color="k", ls="--",
                    label=f"mean {np.mean(ious):.2f}")
    axes[0].set_xlabel("control IoU with the real union")
    axes[0].set_ylabel("images")
    axes[0].set_title("Control strength\n(near 0 = the control is a clean contrast)")
    axes[0].legend(fontsize=9)

    cb = [r["casc"]["abs_rel"] - r["base"]["abs_rel"] for r in rows]
    ct = [r["ctrl"]["abs_rel"] - r["base"]["abs_rel"] for r in rows]
    axes[1].scatter(ct, cb, s=12, alpha=0.5, color=plt.get_cmap("tab10")(1))
    lim = [min(min(cb), min(ct)), max(max(cb), max(ct))]
    axes[1].plot(lim, lim, "k--", lw=1, label="casc == ctrl")
    axes[1].axhline(0, color="gray", lw=0.8)
    axes[1].axvline(0, color="gray", lw=0.8)
    axes[1].set_xlabel("control effect  (ctrl - base, abs_rel)")
    axes[1].set_ylabel("cascade effect  (casc - base, abs_rel)")
    axes[1].set_title("Above the line = segmentation beats a\nmatched-area mask. Below = it does not.")
    axes[1].legend(fontsize=9)
    axes[1].grid(alpha=0.3)

    fig.suptitle("Control diagnostics -- read before quoting any cascade result", fontsize=13)
    fig.tight_layout()
    p = os.path.join(output_dir, "cascade_control_check.png")
    fig.savefig(p, dpi=110, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {p}")


def print_verdict(result: Dict[str, Any], labels: Dict[str, str]):
    ps = result["pixel_stats"]
    print("\n" + "=" * 72)
    print(f"CASCADE VERDICT -- {result['n_samples']} images, {result['n_folds']}-fold")
    print("=" * 72)
    print(f"mean SAM2 union coverage : {ps['mean_coverage'] * 100:.1f}% of pixels")
    print(f"mean control IoU         : {ps['mean_ctrl_iou']:.3f}"
          + ("   <-- HIGH: control is weak, treat casc-vs-ctrl with caution"
             if ps["mean_ctrl_iou"] > 0.5 else ""))
    if ps["mean_coverage"] > 0.8:
        print("  !! coverage > 80%: the cascade masks out almost nothing, so it is\n"
              "     nearly identical to the depth-only baseline. Tighten\n"
              "     --min-mask-area / --max-coverage and re-run.")

    print("\nWhole-image abs_rel (lower better):")
    for arm in ARMS:
        s = result["summary"]["abs_rel"]
        print(f"  {labels[arm]:<28} {s[f'{arm}_mean']:.4f} +/- {s[f'{arm}_std']:.4f}")

    s = result["summary"]["abs_rel"]
    print(f"\ncascade vs depth-only : {s['casc_vs_base_mean']:+.4f} abs_rel "
          f"({s['casc_vs_base_improved']} improved / {s['casc_vs_base_worsened']} worsened)")
    print(f"control vs depth-only : {s['ctrl_vs_base_mean']:+.4f} abs_rel "
          f"({s['ctrl_vs_base_improved']} improved / {s['ctrl_vs_base_worsened']} worsened)")
    print(f"CASCADE vs CONTROL    : {s['casc_vs_ctrl_mean']:+.4f} abs_rel "
          f"({s['casc_vs_ctrl_improved']} improved / {s['casc_vs_ctrl_worsened']} worsened)")
    print("\n  Negative = fewer errors. The last line is the thesis answer: if it is\n"
          "  not clearly negative with a healthy improved/worsened margin, the\n"
          "  cascade's gain is a masking artefact, not segmentation.\n")


# =========================================================================
# Driver
# =========================================================================
def probe_split_size(split: str) -> Optional[int]:
    """Examples actually reachable in `split`, or None if it cannot be determined.

    Worth calling before committing to a split budget, because the number is NOT
    what the dataset card advertises. `sayakpaul/nyu_depth_v2` ships a loading
    script over 12 x 3GB tarballs (~47.5k images), but `datasets>=4.0` removed
    script support and raises `RuntimeError: Dataset scripts are no longer
    supported`, so this module's fallback loads the auto-converted parquet branch
    instead -- which is PARTIAL. Measured here: train holds 3,100 images and
    validation 654. The card's 47,584 is unreachable from that path.

    Counted by reading the parquet FOOTERS (a few KB at the end of each shard via
    HTTP range requests), so this costs seconds and downloads no image data. The
    streaming API itself carries no count -- `ds.info.splits` is empty -- which is
    why the footers are read instead.

    Returns None if the parquet branch is unavailable (e.g. on a datasets
    version old enough to run the loading script, where the tarball count
    applies and is not cheaply knowable).
    """
    try:
        import fsspec
        import pyarrow.parquet as pq
        base = (f"hf://datasets/{NYU_DATASET}@refs/convert/parquet/"
                f"default/partial-{split}")
        fs = fsspec.filesystem("hf")
        files = sorted(fs.glob(base + "/*.parquet"))
        if not files:
            return None
        total = 0
        for f in files:
            with fs.open(f, "rb") as fh:
                total += pq.ParquetFile(fh).metadata.num_rows
        return total
    except Exception:
        return None


def check_split_budget(split: str, eval_offset: int, num_samples: int) -> None:
    """Fail before a multi-hour run if the split cannot supply
    [eval_offset, eval_offset + num_samples)."""
    size = probe_split_size(split)
    if size is None:
        print(f"note: could not determine the size of '{split}' up front -- the run "
              f"will stop early if it runs out of images.")
        return
    need = eval_offset + num_samples
    print(f"'{split}' holds {size} images; this run needs [{eval_offset}:{need}].")
    if need > size:
        raise ValueError(
            f"'{split}' has only {size} images, but --eval-offset {eval_offset} + "
            f"--num-samples {num_samples} needs {need}. Lower --num-samples to "
            f"{size - eval_offset}, or lower --eval-offset (which must stay >= the "
            f"--ft-n the checkpoint was fine-tuned with).")


def check_split_budget(split: str, eval_offset: int, num_samples: int) -> None:
    """Fail before a multi-hour run if the split cannot supply
    [eval_offset, eval_offset + num_samples)."""
    size = probe_split_size(split)
    if size is None:
        print(f"note: could not determine the size of '{split}' up front -- the run "
              f"will stop early if it runs out of images.")
        return
    need = eval_offset + num_samples
    print(f"'{split}' holds {size} images; this run needs [{eval_offset}:{need}].")
    if need > size:
        raise ValueError(
            f"'{split}' has only {size} images, but --eval-offset {eval_offset} + "
            f"--num-samples {num_samples} needs {need}. Lower --num-samples to "
            f"{size - eval_offset}, or lower --eval-offset (which must stay >= the "
            f"--ft-n the checkpoint was fine-tuned with).")


def check_ft_contract(baseline_slug: str, split: str, eval_offset: int) -> None:
    """If `baseline_slug` is a local fine-tuned checkpoint written by
    finetune_backbone.py, verify this run's split+offset actually exclude the
    images that checkpoint trained on. Silent for stock HF checkpoints.

    The check exists because the failure is invisible otherwise: evaluating a
    checkpoint on its own training images produces excellent numbers, not an error.
    """
    meta_path = os.path.join(baseline_slug, "ft_meta.json")
    if not os.path.isfile(meta_path):
        return
    with open(meta_path) as fh:
        meta = json.load(fh)
    ft_n = int(meta.get("ft_n", 0))
    ft_split = meta.get("split", "train")
    print(f"fine-tuned checkpoint: trained on {ft_split}[0:{ft_n}] "
          f"(from {meta.get('model_id')}, {meta.get('max_steps')} steps, "
          f"trainable={meta.get('trainable')})")
    if not meta.get("complete", True):
        print("  WARNING: this checkpoint was saved mid-schedule (save_every), not "
              "at the end of training -- the LR never finished annealing. Usable, "
              "but not the final model.")
    if ft_split == split and eval_offset < ft_n:
        raise ValueError(
            f"LEAKAGE: this checkpoint trained on {ft_split}[0:{ft_n}] but the run "
            f"evaluates {split}[{eval_offset}:...] -- images [{eval_offset}:{ft_n}] "
            f"are in BOTH sets. Pass --eval-offset {ft_n} (or higher).")
    print(f"disjointness OK: evaluating {split}[{eval_offset}:...], "
          f"checkpoint saw {ft_split}[0:{ft_n}]")


def run_kfold(num_samples: int = 3000, n_folds: int = 5, split: str = "train",
              baseline_slug: str = "da2-metric-small", sam2_size: str = "small",
              output_dir: str = "seg_cascade_out", device: Optional[str] = None,
              max_depth: float = 10.0, seed: int = 42, sam2_quick_test: bool = False,
              checkpoints_dir: str = "checkpoints", relative: bool = False,
              align_region: str = "all", min_mask_area: float = 0.0,
              max_coverage: float = 1.0, labels: Optional[Dict[str, str]] = None,
              eval_offset: int = 0) -> Dict[str, Any]:
    """Inference (resumable) -> 5-fold aggregation -> CSVs + plots.

    eval_offset -- skip the first `eval_offset` items of `split`. Set it to the
    --ft-n a fine-tuned checkpoint was trained with; the two sets are then
    disjoint and no NYU validation images are needed.
    """
    os.makedirs(output_dir, exist_ok=True)
    if eval_offset < 0:
        raise ValueError(f"eval_offset must be >= 0, got {eval_offset}")
    if eval_offset and split != "train":
        raise ValueError(f"eval_offset only makes sense on the train split "
                         f"(a fine-tune pool carved out of it), got split={split!r}")
    check_ft_contract(baseline_slug, split, eval_offset)
    check_split_budget(split, eval_offset, num_samples)
    relative = relative or (baseline_slug in K.RELATIVE_SLUGS)
    jsonl = run_inference(num_samples, split, baseline_slug, sam2_size, output_dir,
                          device, max_depth, sam2_quick_test, checkpoints_dir,
                          relative=relative, align_region=align_region,
                          min_mask_area=min_mask_area, max_coverage=max_coverage,
                          seed=seed, eval_offset=eval_offset)
    result = aggregate(jsonl, n_folds, seed, output_dir, baseline_slug,
                       relative=relative, align_region=align_region)
    write_csvs(result, output_dir)
    labels = labels or {"base": "depth-only", "casc": "cascade (SAM2 gate)",
                        "ctrl": "control (rolled mask)"}
    plot_three_arm(result, output_dir, labels)
    plot_region_three(result, output_dir, labels)
    plot_control_check(result, output_dir, jsonl)
    print_verdict(result, labels)
    return result


# =========================================================================
# Self-test: mask/region logic only -- no torch, no GPU, no download
# =========================================================================
def _selftest() -> int:
    """Exercises the pure-numpy mask machinery. Runs anywhere."""
    rng = np.random.default_rng(0)
    img = rng.integers(0, 255, (60, 80, 3), dtype=np.uint8)

    # a plausible union: two blobs
    union = np.zeros((60, 80), dtype=bool)
    union[10:30, 10:40] = True
    union[35:55, 45:70] = True

    # --- apply_region_fill changes exactly the region, and only that ---------
    filled = apply_region_fill(img, union)
    changed = np.any(filled != img, axis=2)
    assert changed.sum() == union.sum(), (changed.sum(), union.sum())
    assert np.array_equal(changed, union), "fill touched pixels outside the region"
    assert (filled[union] == FILL_VALUE).all(), "filled pixels are not the fill value"
    assert np.array_equal(filled[~union], img[~union]), "unmasked pixels were altered"

    # --- control_region preserves area exactly, reports IoU in [0,1] ---------
    for trial in range(5):
        ctrl, iou = control_region(union, np.random.default_rng(trial))
        assert ctrl.sum() == union.sum(), (ctrl.sum(), union.sum())
        assert ctrl.shape == union.shape
        assert 0.0 <= iou <= 1.0, iou
        # lowest-IoU candidate selection must beat a plain zero shift
        zero_iou = (np.logical_and(union, union).sum() / np.logical_or(union, union).sum())
        assert iou <= zero_iou + 1e-12, "control is no better than an unshifted copy"

    # --- control is reproducible for a given sample index -------------------
    a, _ = control_region(union, np.random.default_rng([42, 7]))
    b, _ = control_region(union, np.random.default_rng([42, 7]))
    assert np.array_equal(a, b), "control is not reproducible from its seed"

    # --- erode_region is a strict subset, and never grows -------------------
    inner = erode_region(union, frac=0.05)
    assert not (inner & ~union).any(), "erosion produced pixels outside the region"
    assert inner.sum() < union.sum(), "erosion did not shrink a 20x30 blob"
    assert erode_region(union, frac=0.0).sum() == union.sum(), "frac=0 must be identity"

    # --- select_masks honours the filters ----------------------------------
    def _md(x0, x1, y0, y1):
        seg = np.zeros((60, 80), dtype=bool)
        seg[y0:y1, x0:x1] = True
        return {"segmentation": seg, "area": int(seg.sum())}

    masks = [_md(0, 20, 0, 20), _md(40, 60, 0, 20), _md(0, 80, 30, 60)]
    full, n_full = select_masks(masks, (60, 80))
    assert n_full == 3 and full.sum() > 0
    # max_coverage below the big mask's share must drop the big mask
    capped, n_capped = select_masks(masks, (60, 80), max_coverage=0.3)
    assert n_capped < n_full, "coverage cap kept everything"
    assert capped.mean() <= 0.3, capped.mean()
    # min_mask_area filters small masks: 0.1*4800 = 480px drops both 400px blobs,
    # leaving only the 2400px one.
    _small, n_small = select_masks(masks, (60, 80), min_mask_area=0.1)
    assert n_small == 1, f"min_mask_area should leave exactly the large mask, got {n_small}"

    # --- metrics_in_region returns None on an empty region ------------------
    p = np.full((60, 80), 2.0, dtype=np.float32)
    g = np.full((60, 80), 2.0, dtype=np.float32)
    assert K.metrics_in_region(p, g, np.zeros((60, 80), bool)) is None

    # --- the leakage guard actually fires -----------------------------------
    # Evaluating a fine-tuned checkpoint on its own training prefix is the one
    # mistake that produces good numbers instead of an error, so assert on it.
    import tempfile
    with tempfile.TemporaryDirectory() as td:
        with open(os.path.join(td, "ft_meta.json"), "w") as fh:
            json.dump({"model_id": "x", "ft_n": 2000, "split": "train",
                       "trainable": ["neck", "head"], "max_steps": 10}, fh)
        check_ft_contract(td, "train", 2000)          # exactly disjoint -> OK
        check_ft_contract(td, "train", 5000)          # beyond the pool -> OK
        check_ft_contract(td, "validation", 0)        # other split -> OK
        for bad in (0, 1999):
            try:
                check_ft_contract(td, "train", bad)
            except ValueError:
                pass
            else:
                raise AssertionError(f"leakage not caught for eval_offset={bad}")
    # a stock HF checkpoint has no ft_meta.json -> silently fine
    check_ft_contract("depth-anything/Depth-Anything-V2-Metric-Indoor-Small-hf",
                      "train", 0)

    print("selftest OK -- fill, control, erosion, mask selection, region metrics "
          "and the fine-tune leakage guard all pass.")
    return 0


def main():
    ap = argparse.ArgumentParser(
        description="5-fold CV of a SAM2 masked-input depth cascade vs depth-only vs a "
                    "mask-matched control, on NYU Depth V2.")
    ap.add_argument("--selftest", action="store_true",
                    help="run the CPU-only mask-logic self-test and exit (no models)")
    ap.add_argument("--num-samples", type=int, default=3000)
    ap.add_argument("--n-folds", type=int, default=5)
    ap.add_argument("--split", default="train",
                    help="'train' (NYU val has only ~654 images; still fair -- see docstring)")
    ap.add_argument("--baseline", default="da2-metric-small",
                    help="preset slug or a HuggingFace id / local metric checkpoint path")
    ap.add_argument("--sam2-size", default="small", choices=sorted(K.SAM2_PRESETS))
    ap.add_argument("--output-dir", default="seg_cascade_out")
    ap.add_argument("--checkpoints-dir", default="checkpoints")
    ap.add_argument("--device", default=None)
    ap.add_argument("--max-depth", type=float, default=10.0)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--sam2-quick-test", action="store_true",
                    help="sparser SAM2 grid -- pair with --num-samples 20 for a smoke test")
    ap.add_argument("--align-region", default="all", choices=["all", "inside"],
                    help="relative backbones only: fit the scale+shift on all pixels "
                         "(existing protocol) or on the scored union only")
    ap.add_argument("--min-mask-area", type=float, default=0.0,
                    help="drop SAM2 masks smaller than this fraction of the image")
    ap.add_argument("--max-coverage", type=float, default=1.0,
                    help="cap the union at this fraction of the image by dropping the "
                         "largest masks (guards against a degenerate near-full union)")
    ap.add_argument("--eval-offset", type=int, default=0,
                    help="skip the first N items of the split. Set to the --ft-n a "
                         "fine-tuned checkpoint was trained with so the evaluated "
                         "images are disjoint from its training images.")
    a = ap.parse_args()

    if a.selftest:
        raise SystemExit(_selftest())

    run_kfold(num_samples=a.num_samples, n_folds=a.n_folds, split=a.split,
              baseline_slug=a.baseline, sam2_size=a.sam2_size,
              output_dir=a.output_dir, device=a.device, max_depth=a.max_depth,
              seed=a.seed, sam2_quick_test=a.sam2_quick_test,
              checkpoints_dir=a.checkpoints_dir, align_region=a.align_region,
              min_mask_area=a.min_mask_area, max_coverage=a.max_coverage,
              eval_offset=a.eval_offset)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile finetune_backbone.py
#!/usr/bin/env python3
"""
finetune_backbone.py

Backbone-agnostic HEAD-ONLY fine-tuning of a monocular depth model on a BOUNDED,
DETERMINISTIC slice of NYU Depth V2 train, so the result can be evaluated by
seg_cascade_kfold.py on a DISJOINT slice of the same split.

Generalizes finetune_midas.py (which is hardcoded to Intel/dpt-large) in two ways:

  1. ANY transformers depth checkpoint -- --model-id works for the
     Depth-Anything-V2 family, MiDaS/DPT, and anything else exposing
     AutoModelForDepthEstimation + AutoImageProcessor.

  2. A BOUNDED train pool instead of an infinite shuffle over the whole split.

WHY THE POOL IS BOUNDED -- THIS IS THE WHOLE POINT
--------------------------------------------------
finetune_midas.py trained on an infinite shuffle of the ENTIRE NYU train split.
That makes every NYU train image training data, so the only legal evaluation is
the 654-image validation split. This script instead trains on a fixed PREFIX of
the split, which leaves the rest of train legally available for evaluation:

    NYU train split
    |---[0 : ft_n]---+---[ft_n : ft_n + num_eval]---|
        FINE-TUNED        EVALUATED, never trained on
        never scored

Run the evaluation with --eval-offset ft_n (seg_cascade_kfold.py --eval-offset)
and the two sets are disjoint by construction: no NYU validation images are
needed, and the evaluation set can be 3000 images instead of 654.

The pool is a contiguous PREFIX, not a random sample of the split, precisely so
the evaluation can exclude it with a single offset. Within the prefix it is
shuffled, so training still sees varied ordering and epochs differ.

TWO KINDS OF FINE-TUNE -- DO NOT CONFLATE THEM IN A RESULTS TABLE
-----------------------------------------------------------------
  conversion  -- MiDaS / DA-V2-relative output unitless disparity, which the
                 metrics cannot score at all. Fine-tuning makes them METRIC.
                 It changes what the model OUTPUTS.
  adaptation  -- DA-V2-metric checkpoints already emit meters (trained on
                 Hypersim). Fine-tuning them on NYU is DOMAIN ADAPTATION.
                 It changes how ACCURATE the model is.

Both are supported here, but they support different claims. Say which is which.

SCOPE -- HEAD-ONLY
------------------
Everything is frozen except parameters whose name matches --trainable
(default "neck"/"head"). The script RAISES if nothing matches rather than
silently training nothing, and prints the candidate module names so you can pick
a better pattern for an unfamiliar architecture.

LEAKAGE CHECKLIST
-----------------
  * train on train[0 : ft_n]
  * evaluate with --eval-offset ft_n, and eval_offset + num_samples <= split size
  * do NOT point --model-id at a checkpoint that was itself trained on NYU train
    (e.g. Intel/zoedepth-nyu-kitti) -- that is leakage this script cannot detect
  * keep ft_n, eval_offset and the checkpoint together -- a model fine-tuned with
    one offset is invalid for an evaluation using another

USAGE
-----
python finetune_backbone.py --model-id depth-anything/Depth-Anything-V2-Metric-Indoor-Small-hf \
        --ft-n 2000 --out-dir ft_da2_metric_small
python seg_cascade_kfold.py --baseline ft_da2_metric_small \
        --split train --eval-offset 2000 --num-samples 3000 --n-folds 5
"""
import argparse
import os
from typing import Any, Dict, Iterator, List, Optional, Sequence

import numpy as np
import torch
import torch.nn.functional as F

NYU_DATASET = "sayakpaul/nyu_depth_v2"
MIN_DEPTH, MAX_DEPTH = 1e-3, 10.0     # NYU indoor valid depth band (matches seg_depth_kfold)


# =========================================================================
# NYU loader. Mirrors the datasets>=4 parquet fallback in seg_depth_kfold so
# the fine-tune and the evaluation read the split the SAME way -- if their
# orderings ever diverged, the offset would no longer guarantee disjointness.
# =========================================================================
def _load_stream(split: str):
    from datasets import load_dataset
    try:
        return load_dataset(NYU_DATASET, split=split, streaming=True)
    except RuntimeError:
        data_files = {split: f"hf://datasets/{NYU_DATASET}@refs/convert/parquet/"
                              f"default/partial-{split}/*.parquet"}
        return load_dataset("parquet", data_files=data_files, split=split, streaming=True)


def split_size(split: str) -> Optional[int]:
    """Examples actually reachable in `split`, or None if undeterminable.

    NOT the number the dataset card advertises. `sayakpaul/nyu_depth_v2` ships a
    loading script over 12 x 3GB tarballs (~47.5k images), but `datasets>=4.0`
    removed script support, so the loader below falls back to the auto-converted
    parquet branch -- which is PARTIAL: measured, train holds 3,100 images and
    validation 654.

    Counted from the parquet FOOTERS (a few KB per shard over HTTP range
    requests), so it is fast and downloads no image data. The streaming API
    carries no count, which is why the footers are read directly.

    Mirrors probe_split_size() in seg_cascade_kfold.py -- the two must agree, or
    the offset would stop guaranteeing that the fine-tune and the evaluation sets
    are disjoint. Kept as a copy so this script needs no cv2 import.
    """
    try:
        import fsspec
        import pyarrow.parquet as pq
        base = (f"hf://datasets/{NYU_DATASET}@refs/convert/parquet/"
                f"default/partial-{split}")
        fs = fsspec.filesystem("hf")
        files = sorted(fs.glob(base + "/*.parquet"))
        if not files:
            return None
        total = 0
        for f in files:
            with fs.open(f, "rb") as fh:
                total += pq.ParquetFile(fh).metadata.num_rows
        return total
    except Exception:
        return None


def nyu_train_pool(ft_n: int, seed: int, buffer_size: int = 1000,
                   epoch: int = 0) -> Iterator[Dict[str, Any]]:
    """One epoch over EXACTLY the first `ft_n` items of the train split,
    shuffled within the pool (seed varies per epoch).

    `take(ft_n)` before `shuffle` is deliberate: the pool must stay a contiguous
    PREFIX so the evaluation can exclude it with --eval-offset ft_n. Shuffling
    the whole split first would draw a scattered subset that no offset can
    exclude.
    """
    ds = _load_stream("train").take(ft_n)
    return iter(ds.shuffle(seed=seed + epoch, buffer_size=max(1, min(buffer_size, ft_n))))


# =========================================================================
# Model: load trainable, freeze everything but the --trainable name matches
# =========================================================================
def load_trainable(model_id: str, device: str, patterns: Sequence[str]):
    """Returns (model in train mode on device, processor). HEAD-ONLY: only
    parameters whose name contains one of `patterns` keep requires_grad."""
    from transformers import AutoImageProcessor, AutoModelForDepthEstimation
    processor = AutoImageProcessor.from_pretrained(model_id)
    model = AutoModelForDepthEstimation.from_pretrained(model_id)

    n_train = n_total = 0
    for name, p in model.named_parameters():
        p.requires_grad = any(pat in name for pat in patterns)
        n_total += p.numel()
        if p.requires_grad:
            n_train += p.numel()
    if n_train == 0:
        tops = sorted({n.split(".")[0] for n, _ in model.named_parameters()})
        raise RuntimeError(
            f"No parameters matched {list(patterns)} for {model_id}. "
            f"Top-level modules are {tops} -- pass a --trainable pattern that "
            f"names the part you want to train.")
    model.to(device).train()
    print(f"trainable: {n_train/1e6:.1f}M / {n_total/1e6:.1f}M "
          f"({100*n_train/n_total:.1f}%) -- matched {list(patterns)}, rest frozen")
    return model, processor


# =========================================================================
# Scale-Invariant Log loss (Eigen et al.; BTS variance_focus=0.85, scale=10).
# Used for BOTH kinds of fine-tune: it drives a relative model toward meters,
# and it is a sound metric loss for an already-metric model.
# =========================================================================
def silog_loss(pred: torch.Tensor, target: torch.Tensor,
               mask: torch.Tensor, lam: float = 0.85) -> torch.Tensor:
    pred = pred.clamp(min=MIN_DEPTH)
    d = torch.log(pred[mask]) - torch.log(target[mask])
    # mean(d^2) - lam*mean(d)^2 >= (1-lam)*var >= 0, so the sqrt is safe.
    return torch.sqrt((d ** 2).mean() - lam * (d.mean() ** 2) + 1e-7) * 10.0


def make_batch(items: List[Dict[str, Any]], processor, device: str):
    """PIL images -> pixel_values; depth_map -> (B,H,W) meters + valid mask."""
    imgs, gts = [], []
    for it in items:
        imgs.append(it["image"].convert("RGB"))
        gts.append(torch.from_numpy(np.array(it["depth_map"]).astype(np.float32)))
    pixel_values = processor(images=imgs, return_tensors="pt")["pixel_values"].to(device)
    gt = torch.stack(gts).to(device)
    mask = (gt > MIN_DEPTH) & (gt < MAX_DEPTH) & torch.isfinite(gt)
    return pixel_values, gt, mask


def _write_meta(out_dir: str, model_id: str, ft_n: int, max_steps: int,
                batch_size: int, seed: int, patterns: Sequence[str],
                complete: bool) -> None:
    """Record the fine-tune contract inside the checkpoint directory, so an
    evaluation pointed at this folder can VERIFY its offset instead of trusting a
    notebook variable. `complete` distinguishes a finished run from a
    save_every checkpoint of one that died partway."""
    import json
    with open(os.path.join(out_dir, "ft_meta.json"), "w") as fh:
        json.dump({"model_id": model_id, "ft_n": ft_n, "split": "train",
                   "trainable": list(patterns), "max_steps": max_steps,
                   "batch_size": batch_size, "seed": seed,
                   "complete": bool(complete)}, fh, indent=2)


def train(out_dir: str, model_id: str, ft_n: int, max_steps: int, batch_size: int,
          lr: float, seed: int, buffer_size: int, device: str, log_every: int,
          save_every: int, patterns: Sequence[str]):
    """Step-based head-only training over the bounded pool. The pool is re-created
    (reshuffled) each epoch, so `max_steps` can exceed one pass over it."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    size = split_size("train")
    if size is not None and ft_n > size:
        raise ValueError(
            f"--ft-n {ft_n} exceeds the train split size ({size}). The pool is a "
            f"prefix of the split, so it cannot be larger than the split. Lower "
            f"--ft-n to leave room for the evaluation too: the eval needs "
            f"[ft_n, ft_n + num_samples) to fit inside the same {size} images.")
    if size is not None:
        print(f"train split size: {size} -> fine-tune pool [0:{ft_n}], "
              f"evaluation may use [{ft_n}:{size}]")
    else:
        print("train split size unknown -- will verify the pool length by counting "
              "the first epoch")

    model, processor = load_trainable(model_id, device, patterns)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, total_steps=max_steps, pct_start=0.1, anneal_strategy="cos")
    use_amp = (device == "cuda")
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    os.makedirs(out_dir, exist_ok=True)
    # Written BEFORE training, not after: save_every may leave a usable
    # checkpoint in this directory if the session dies, and the leakage guard in
    # seg_cascade_kfold keys off this file. Without it a mid-run checkpoint would
    # look like a stock model and could be evaluated at the wrong offset silently.
    _write_meta(out_dir, model_id, ft_n, max_steps, batch_size, seed, patterns,
                complete=False)
    running: List[float] = []
    drawn_per_epoch = 0
    epoch = 0
    stream = nyu_train_pool(ft_n, seed, buffer_size, epoch)

    for step in range(1, max_steps + 1):
        items = []
        while len(items) < batch_size:
            try:
                items.append(next(stream))
                drawn_per_epoch += 1
            except StopIteration:
                # pool exhausted -> next epoch (reshuffled)
                epoch += 1
                print(f"  -- epoch {epoch} over the {ft_n}-image pool "
                      f"(each epoch = {drawn_per_epoch} images) --")
                if epoch == 1 and size is None and drawn_per_epoch < ft_n:
                    raise ValueError(
                        f"the train pool yielded only {drawn_per_epoch} images, but "
                        f"--ft-n {ft_n} was requested. The split is smaller than "
                        f"assumed, so the evaluation offset {ft_n} would run off "
                        f"the end. Lower --ft-n below {drawn_per_epoch}.")
                drawn_per_epoch = 0
                stream = nyu_train_pool(ft_n, seed, buffer_size, epoch)
        pixel_values, gt, mask = make_batch(items, processor, device)
        if int(mask.sum()) == 0:
            continue

        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            pred = model(pixel_values=pixel_values).predicted_depth
            pred = F.interpolate(pred.unsqueeze(1), size=gt.shape[-2:],
                                 mode="bicubic", align_corners=False).squeeze(1)
        # SILog in fp32 -- log/sqrt are unstable under autocast fp16.
        loss = silog_loss(pred.float(), gt.float(), mask)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 5.0)
        scaler.step(opt)
        scaler.update()
        sched.step()

        running.append(loss.item())
        if step % log_every == 0:
            with torch.no_grad():
                p, g = pred.float()[mask].clamp(min=MIN_DEPTH), gt.float()[mask]
                absrel = (torch.abs(p - g) / g).mean().item()
            print(f"step {step:5d}/{max_steps} | silog {np.mean(running[-log_every:]):.4f} "
                  f"| train AbsRel {absrel:.4f} | lr {sched.get_last_lr()[0]:.2e}")
        if save_every and step % save_every == 0:
            model.save_pretrained(out_dir)
            processor.save_pretrained(out_dir)
            print(f"  checkpoint -> {out_dir} (step {step})")

    model.save_pretrained(out_dir)
    processor.save_pretrained(out_dir)
    # Mark the run finished. seg_cascade_kfold warns if this is False, because a
    # partial checkpoint is still usable but was saved mid-schedule.
    _write_meta(out_dir, model_id, ft_n, max_steps, batch_size, seed, patterns,
                complete=True)
    print(f"\ndone -- fine-tuned model saved to '{out_dir}'")
    print(f"CONTRACT: this model trained on train[0:{ft_n}]. Evaluate it with:\n"
          f"  python seg_cascade_kfold.py --baseline {out_dir} --split train "
          f"--eval-offset {ft_n} --num-samples <n> --n-folds 5")


def main():
    ap = argparse.ArgumentParser(
        description="Head-only fine-tune a depth model on a bounded NYU train pool, "
                    "leaving a disjoint slice for evaluation.")
    ap.add_argument("--model-id", default="Intel/dpt-large",
                    help="any HF depth checkpoint (DA-V2 family, MiDaS/DPT, ...)")
    ap.add_argument("--ft-n", type=int, default=2000,
                    help="size of the fine-tune pool = train[0:ft_n]. The evaluation "
                         "must then run with --eval-offset ft_n.")
    ap.add_argument("--out-dir", default="ft_backbone",
                    help="where to save_pretrained the fine-tuned model")
    ap.add_argument("--max-steps", type=int, default=1500)
    ap.add_argument("--batch-size", type=int, default=4, help="lower to 2 if a T4 OOMs")
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--buffer-size", type=int, default=1000, help="streaming shuffle buffer")
    ap.add_argument("--trainable", default="neck,head",
                    help="comma-separated name substrings to keep trainable "
                         "(default: neck,head -- head-only)")
    ap.add_argument("--device", default=None, help="cuda | cpu (auto if omitted)")
    ap.add_argument("--log-every", type=int, default=25)
    ap.add_argument("--save-every", type=int, default=500, help="0 to save only at the end")
    a = ap.parse_args()

    device = a.device or ("cuda" if torch.cuda.is_available() else "cpu")
    patterns = [p.strip() for p in a.trainable.split(",") if p.strip()]
    print(f"device={device} | model={a.model_id} | pool=train[0:{a.ft_n}] | "
          f"{a.max_steps} steps x batch {a.batch_size} | trainable={patterns}")
    train(a.out_dir, a.model_id, a.ft_n, a.max_steps, a.batch_size, a.lr, a.seed,
          a.buffer_size, device, a.log_every, a.save_every, patterns)


if __name__ == "__main__":
    main()

In [ ]:
import importlib, json, os
import seg_depth_kfold as K
import seg_cascade_kfold as C
importlib.reload(K)  # pick up whatever the %%writefile cells just wrote to disk
importlib.reload(C)

assert C.K is K, "the cascade module did not bind to seg_depth_kfold"
for sym in ("score_sample_three", "control_region", "select_masks", "erode_region",
            "apply_region_fill", "aggregate", "run_kfold", "_align_arm",
            "probe_split_size", "check_ft_contract", "check_split_budget"):
    assert hasattr(C, sym), f"stale seg_cascade_kfold -- missing {sym}"
assert C.ARMS == ("base", "casc", "ctrl"), C.ARMS
assert C.KEY_PAIR == ("casc", "ctrl"), C.KEY_PAIR

print("loaded seg_cascade_kfold | arms:", C.ARMS, "| key contrast:", C.KEY_PAIR,
      "| fill value:", C.FILL_VALUE)
print("shared baselines:", len(K.BASELINE_IDS), "| SAM2 sizes:", sorted(K.SAM2_PRESETS))

## 2. Self-test (no GPU, no models, seconds)

Exercises the mask machinery — fill, control roll, erosion, mask filtering — the shared region metrics, and the fine-tune leakage guard. **Run this first**; if it fails, nothing below is meaningful.

In [ ]:
assert C._selftest() == 0

## 3. Stage 1 — fine-tune (head-only, bounded pool)

`finetune_backbone.py` is written below and then run in-process. It freezes everything but the parameters matching `TRAINABLE` and trains with the **SILog** loss (Eigen et al.; the BTS variant, variance-focus 0.85, ×10) — the same protocol as the repo's existing `finetune_midas.py`, generalized to any HuggingFace depth checkpoint.

**The pool is a contiguous prefix, not a random sample.** That is what lets stage 2 exclude it with a single offset. Within the prefix it is shuffled each epoch, so training still sees varied ordering.

**This is head-only.** The ViT backbone stays frozen, so what adapts is the fusion head, not the features. The cell prints the trainable fraction — if it is ~0%, the run aborts rather than silently training nothing.

### 3a. Measure the split, then fix the budget

**Do not skip this cell.** The `sayakpaul/nyu_depth_v2` card advertises 47,584 training images, but that comes from 12 × 3 GB tarballs read by a **loading script**, and `datasets>=4.0` removed script support. The loaders in this repo therefore fall back to the auto-converted parquet branch, which is **partial**: it holds **3,100** train images. The probe below reads the parquet footers (a few KB per shard, no image downloads) and reports the real number, then refuses to run if `FT_N + NUM_SAMPLES` does not fit.

**This is why `NUM_SAMPLES` is 2000, not 3000.** With only 3,100 images reachable, a fine-tune pool and a 3,000-image evaluation cannot both exist. The knobs trade one against the other; 1000 + 2000 is the default. If a future dataset revision carries the full split, raise both.

In [ ]:
import os
import seg_cascade_kfold as C

# -----------------------------------------------------------------------
# STAGE 1 + 2 BUDGET -- edit these
# -----------------------------------------------------------------------
SPLIT       = "train"     # the ONLY split with enough images to carve in two
FT_N        = 1000    # fine-tune on train[0:FT_N]
NUM_SAMPLES = 2000   # then evaluate train[FT_N : FT_N + NUM_SAMPLES]
N_FOLDS     = 5
EVAL_OFFSET = FT_N        # stage 2 skips exactly the images stage 1 trained on

# Depth backbone. This is a real HF id (or a local path) -- stage 1 fine-tunes it
# and stage 2 evaluates the FINE-TUNED checkpoint, never the stock one.
MODEL_ID     = "depth-anything/Depth-Anything-V2-Small-hf"
TRAINABLE    = "neck,head"    # head-only; the ViT backbone stays frozen

MAX_STEPS  = 1500
BATCH_SIZE = 4        # drop to 2 if a T4 OOMs
LR         = 1e-4
FT_SEED    = 42
SAVE_EVERY = 500              # checkpoint cadence: survives a dead session

FT_DIR   = "/kaggle/working/ft_da2-relative-small"          # fine-tuned checkpoint lands here
OUTPUT_DIR = "/kaggle/working/seg_cascade_da2-relative-small"

# --- the coverage knobs (see the coverage warning above) --------------------
# Drop SAM2 masks smaller than this FRACTION of the image. 0.0 keeps everything.
MIN_MASK_AREA = 0.0
# Cap the union at this FRACTION of the image by dropping the LARGEST masks
# first (walls/floor/ceiling). 1.0 disables the cap.
# If the smoke test shows coverage above ~0.8, raise this to ~0.6 and re-run.
MAX_COVERAGE  = 1.0

# # Only used if RELATIVE is True -- it is False for a fine-tuned
# checkpoint. Kept for reference.
ALIGN_REGION = "all"
RELATIVE     = False

SAM2_SIZE       = "small"     # "tiny" | "small" | "base_plus" | "large"
SEED            = 42
SAM2_QUICK_TEST = False

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- the budget guard -------------------------------------------------------
_size = C.probe_split_size(SPLIT)
print(f"'{SPLIT}' actually holds {_size} images "
      f"(the dataset CARD claims 47584 -- see the markdown above).")
if _size is not None:
    assert EVAL_OFFSET + NUM_SAMPLES <= _size, (
        f"train[{EVAL_OFFSET}:{EVAL_OFFSET + NUM_SAMPLES}] runs past the end of a "
        f"{_size}-image split. Lower NUM_SAMPLES to at most {_size - EVAL_OFFSET}, "
        f"or lower FT_N -- but FT_N and EVAL_OFFSET must stay equal, or stage 2 "
        f"would score images stage 1 trained on.")
    print(f"budget OK: fine-tune [0:{FT_N}] + evaluate "
          f"[{EVAL_OFFSET}:{EVAL_OFFSET + NUM_SAMPLES}] = "
          f"{EVAL_OFFSET + NUM_SAMPLES}/{_size} images")

print(f"device={DEVICE} | backbone={MODEL_ID} | {N_FOLDS}-fold over {NUM_SAMPLES} "
      f"images | sam2={SAM2_SIZE}")
print(f"fine-tune: {MAX_STEPS} steps x batch {BATCH_SIZE}, trainable='{TRAINABLE}'")

### 3b. Run the fine-tune

Resumable in the coarse sense: if `FT_DIR` already holds a **completed** run it is skipped, and `SAVE_EVERY` leaves a checkpoint if the session dies. A checkpoint saved mid-schedule is marked `complete: false` and stage 2 will warn about it — it is usable, but its learning rate never finished annealing.

In [ ]:
import json, importlib
import finetune_backbone as FT
importlib.reload(FT)

# Skip if a FINISHED run is already on disk (a mid-run save_every checkpoint is
# usable but re-running is cheap insurance if you have session time left).
_meta = os.path.join(FT_DIR, "ft_meta.json")
_done = os.path.isfile(_meta) and json.load(open(_meta)).get("complete", False)

if _done:
    print(f"already fine-tuned -> reusing {FT_DIR}")
else:
    FT.train(
        out_dir=FT_DIR, model_id=MODEL_ID, ft_n=FT_N, max_steps=MAX_STEPS,
        batch_size=BATCH_SIZE, lr=LR, seed=FT_SEED, buffer_size=1000,
        device=DEVICE, log_every=25, save_every=SAVE_EVERY,
        patterns=[p.strip() for p in TRAINABLE.split(",") if p.strip()],
    )

### 3c. Sanity-check the fine-tuned checkpoint **before** the long run

Stage 2 loads the checkpoint through the same `transformers` depth pipeline used for evaluation. That pipeline applies its own post-processing, and for some checkpoints that can re-normalise the output — which would undo the fine-tune and leave a model that still emits relative depth. The check below scores a handful of **held-out** images (from the evaluation range, never seen in training) with **no alignment at all**. If the model is metric, AbsRel is a normal number; if it is still relative, AbsRel is enormous.

**If this fails, stop.** Stage 2 would run for hours and produce meaningless numbers, because `RELATIVE = False` tells it to trust the output as meters.

In [ ]:
import numpy as np
import seg_depth_kfold as K

pipe = K.load_metric_depth_model(FT_DIR, DEVICE)
print(f"loaded {FT_DIR} through the depth pipeline")

rows, cov = [], []
for idx, item in C.iter_nyu(SPLIT, 8, EVAL_OFFSET):
    img = np.array(item["image"].convert("RGB"))
    gt = np.array(item["depth_map"]).astype(np.float32)
    pred = K.predict_depth_meters(pipe, img)
    m = K.compute_depth_metrics(pred, gt, max_depth=10.0)
    if m is not None:
        rows.append(m["abs_rel"])
        cov.append(float(np.median(pred) / max(np.median(gt), 1e-6)))

absrel = float(np.nanmean(rows))
print(f"\nheld-out AbsRel (NO alignment): {absrel:.4f}")
print(f"median(pred)/median(gt) ratio  : {np.nanmean(cov):.3f}  "
      f"(~1.0 means the output is already in metres)")

if absrel > 1.0:
    print("\n!! The checkpoint does not look metric. Either the fine-tune did not")
    print("   take, or the pipeline's post-processing is re-normalising the output.")
    print("   Do NOT run stage 2 with RELATIVE=False on this checkpoint.")
else:
    print("\ncheckpoint emits usable metric depth -- stage 2 may proceed.")
del pipe
K._free_gpu()

## 4. Stage 2 — configuration

Read from the stage-1 budget cell above, so the two stages cannot drift apart: `BASELINE_SLUG` is the **fine-tuned checkpoint directory**, and `EVAL_OFFSET` is exactly `FT_N`.

In [ ]:
# Everything here is derived from the stage-1 cell -- do not re-type these.
BASELINE_SLUG = FT_DIR          # the FINE-TUNED checkpoint, not the stock model
EVAL_OFFSET   = FT_N            # stage 2 skips exactly what stage 1 trained on

# The leakage guard is the one check that must not be missing: without it a
# mis-set offset produces excellent numbers instead of an error. Assert here so
# the failure lands in a cheap cell rather than after hours of inference.
_meta_path = os.path.join(FT_DIR, "ft_meta.json")
assert os.path.isfile(_meta_path), (
    f"no ft_meta.json in {FT_DIR} -- stage 1 did not produce a checkpoint. "
    f"Run the stage-1 cells (3a-3b) first.")
_meta = json.load(open(_meta_path))
assert _meta["ft_n"] == EVAL_OFFSET, (
    f"FT_N={FT_N} but the checkpoint trained on train[0:{_meta['ft_n']}]. "
    f"The offset would score images the model trained on.")
if not _meta.get("complete", True):
    print("WARNING: that checkpoint was saved mid-schedule (save_every), not at "
          "the end of training.")

# # RELATIVE=False is CORRECT here, and it is not a typo. Stage 1
# fine-tuned this relative checkpoint with a metric SILog loss, so it
# now emits metres and must NOT be scale+shift aligned -- aligning a
# metric prediction against GT would leak GT into the score and make
# the result incomparable to the other backbones.
RELATIVE = False

print(f"baseline = {BASELINE_SLUG}")
print(f"evaluating {SPLIT}[{EVAL_OFFSET}:{EVAL_OFFSET + NUM_SAMPLES}] "
      f"({NUM_SAMPLES} images the checkpoint never trained on)")

## 5. Smoke test — **run this before the full run**

8 held-out images, sparse SAM2 grid. Its only job is to answer *"is the SAM2 union covering so much of the frame that the cascade is degenerate?"* — plus catch schema/NaN problems early. The JSONL it writes is reused by the full run (already-scored indices are skipped), so nothing is wasted.

Small on purpose: it is the first time the fine-tuned checkpoint, SAM2 and the three-arm scoring all run together.

In [ ]:
smoke = C.run_kfold(
    num_samples=8, n_folds=5, split=SPLIT, baseline_slug=BASELINE_SLUG,
    sam2_size=SAM2_SIZE, output_dir=f"{OUTPUT_DIR}_smoke", device=DEVICE,
    max_depth=10.0, seed=SEED, sam2_quick_test=True, relative=RELATIVE,
    align_region=ALIGN_REGION, min_mask_area=MIN_MASK_AREA, max_coverage=MAX_COVERAGE,
    eval_offset=EVAL_OFFSET,
)

### 5b. Coverage gate

**Decide here.** If `mean coverage` is above ~0.8 the cascade masks out almost nothing, so `casc` is nearly identical to `base` by construction. Raise `MIN_MASK_AREA` and/or set `MAX_COVERAGE ~0.6` in the config cell, delete `..._smoke/cascade_per_sample.jsonl`, and re-run. Also check `mean control IoU`: if it is high, the rolled control overlapped the real union heavily and the `casc vs ctrl` contrast is weak.

In [ ]:
import json, numpy as np

with open(f"{OUTPUT_DIR}_smoke/cascade_per_sample.jsonl") as fh:
    rows = [json.loads(l) for l in fh if l.strip()]

cov = np.array([r["coverage"] for r in rows])
iou = np.array([r["ctrl_iou"] for r in rows])
print(f"{len(rows)} images scored")
print(f"union coverage : mean {cov.mean():.3f}  min {cov.min():.3f}  max {cov.max():.3f}")
print(f"control IoU    : mean {iou.mean():.3f}  min {iou.min():.3f}  max {iou.max():.3f}")
print(f"masks kept     : mean {np.mean([r['n_masks'] for r in rows]):.1f}")
print(f"train indices  : {min(r['idx'] for r in rows)}..{max(r['idx'] for r in rows)} "
      f"(must all be >= EVAL_OFFSET = {EVAL_OFFSET})")

assert min(r["idx"] for r in rows) >= EVAL_OFFSET, \
    "scored an image inside the fine-tune pool -- the offset did not take effect"
for r in rows:
    for arm in ("base", "casc", "ctrl"):
        assert all(np.isfinite(list(r[arm].values()))), f"non-finite metric in {arm}"

if cov.mean() > 0.8:
    print("\n!! coverage > 0.8 -- the cascade masks out almost nothing.")
    print("   Set MAX_COVERAGE = 0.6 (and/or raise MIN_MASK_AREA), then re-smoke.")
else:
    print("\ncoverage looks workable -- proceed to the full run.")
if iou.mean() > 0.5:
    print("!! control IoU is high -- the rolled control overlaps the real union,")
    print("   so 'casc vs ctrl' is a weak contrast on these images.")

## 6. Full run

SAM2 masks → union (filtered) → rolled control → three depth forwards (original, union-filled, control-filled) → scored per image → 5-fold aggregation → CSVs + charts.

**Resumable:** if the session dies partway, re-run this cell — already-scored images are skipped. The control shift is seeded per image *index*, not from a running RNG, so a resumed run reproduces the identical control for every image. Rows also record `eval_offset`, and the run **refuses to resume** a JSONL written with a different offset, since those rows would describe a different image set under the same indices.

**Budget note:** the depth model runs three times per image here, so this is roughly 3× the cost of a com1 notebook at the same `NUM_SAMPLES`.

In [ ]:
result = C.run_kfold(
    num_samples=NUM_SAMPLES,
    n_folds=N_FOLDS,
    split=SPLIT,
    baseline_slug=BASELINE_SLUG,
    sam2_size=SAM2_SIZE,
    output_dir=OUTPUT_DIR,
    device=DEVICE,
    max_depth=10.0,
    seed=SEED,
    sam2_quick_test=SAM2_QUICK_TEST,
    relative=RELATIVE,
    align_region=ALIGN_REGION,
    min_mask_area=MIN_MASK_AREA,
    max_coverage=MAX_COVERAGE,
    eval_offset=EVAL_OFFSET,
)
print("\nDone. Outputs in:", OUTPUT_DIR)

## 7. Results

In [ ]:
import os, json
import pandas as pd
from IPython.display import Image, display

with open(f"{OUTPUT_DIR}/cascade_summary.json") as f:
    res = json.load(f)

ps = res["pixel_stats"]
print(f"{res['n_folds']}-fold CV over {res['n_samples']} images | "
      f"baseline={res['baseline_slug']} | mechanism={res['mechanism']}")
print(f"evaluated from index {res.get('eval_offset')} of '{SPLIT}' "
      f"(fine-tune pool was [0:{res.get('eval_offset')}])")
print(f"mean union coverage: {ps['mean_coverage']*100:.1f}% of pixels | "
      f"mean control IoU: {ps['mean_ctrl_iou']:.3f} | "
      f"mean masks kept: {ps['mean_n_masks']:.1f}\n")

print("Cross-validated summary (mean +/- std across folds):")
display(pd.read_csv(f"{OUTPUT_DIR}/cascade_summary.csv"))

print("\nInside / inner / outside the SAM2 union (the cascade's claim lives in 'inside'):")
display(pd.read_csv(f"{OUTPUT_DIR}/cascade_region.csv"))

for fig in ["cascade_three_arm.png", "cascade_region.png", "cascade_control_check.png"]:
    p = f"{OUTPUT_DIR}/{fig}"
    if os.path.exists(p):
        display(Image(p))

## Notes for interpreting the results

### What these numbers are, and are not

- **The checkpoint is fine-tuned on NYU, and scored on 2000 held-out NYU images.** So this is *not* a zero-shot benchmark and is **not comparable** to the `com1` notebooks or to published NYU leaderboards. It answers one question: *given this backbone, does gating its input by SAM2 segmentation beat not gating it?* That comparison is internally valid because all three arms share one checkpoint and one image set.
- ****The fine-tune here is a CONVERSION, not an adaptation.** `depth-anything/Depth-Anything-V2-Small-hf` started out unable to produce metres; the numbers below describe a model that was fine-tuned specifically so it could. Do not present them as the stock checkpoint's performance.**
- **Head-only.** The ViT backbone was frozen, so what adapted is the fusion head. A full fine-tune would likely move these numbers; the ablation's *conclusion* should not depend on that, but the absolute values would.

### The cascade result

- **`CASCADE vs CONTROL` is the thesis answer.** It is the only contrast that survives the masking confound. A negative `casc_vs_ctrl` on `abs_rel`/`rmse` (positive on `delta1`), with a healthy `improved`/`worsened` margin, is evidence that segmentation itself helps. If it is ~0, the cascade's apparent gain belongs to masking, not segmentation — report that honestly; it is a real finding.
- **The cascade can only make a claim *inside* the union.** Outside it, the depth model saw flat gray and its predictions are meaningless, so whole-image means are dragged down by construction. Read `cascade_region.csv`'s `inside` rows as the headline, and `outside` rows as the price of masking.
- **`inner` vs `inside`.** `inner` is the union eroded by 2%. If the effect is much larger `inside` than `inner`, it is concentrated at the gray/object boundary, where the fill is most out-of-distribution, rather than in the object itself.
- **Control strength.** `cascade_control_check.png`'s left panel is the diagnostic: control IoU near 0 means the control was a clean contrast; near 1 means the two arms saw nearly the same pixels and the comparison on those images is void. The right panel plots cascade effect against control effect per image — points **above** the diagonal are images where SAM2's semantics beat a matched-area mask of the same shape.
- **Coverage.** If mean coverage is very high, `casc ≈ base` mechanically. That is a property of SAM2's automatic generator, not a result.
- **Reproducibility:** folds are fixed by `SEED`; the control shift is a deterministic function of `(SEED, image index)`; the fine-tune pool and its shuffle are fixed by `FT_SEED`.

### Housekeeping

- **Download outputs** before the session ends — Kaggle wipes `/kaggle/working`. Keep `cascade_per_sample.jsonl` to resume or re-fold without re-running inference, and keep `FT_DIR` if you want to re-run stage 2 without re-training.
- **If you change `FT_N`, re-run stage 1 too.** The checkpoint's `ft_meta.json` records the pool it trained on, and stage 2 aborts if `EVAL_OFFSET` would score any of it.